## 1- Nicolas script to process jsons

### 1.1 run this before everything else

In [ ]:
!mkdir -p input

import re
tag_pattern = '<[^>]+>'
tag_compiled_pattern = re.compile (tag_pattern)
#tag_element_pattern = '<([^>\s]+)>'
tag_element_pattern = '[^><\s/]+'
tag_element_compiled_pattern = re.compile(tag_element_pattern)

body_element_pattern = '<body'
body_element_compiled_pattern = re.compile(body_element_pattern, re.IGNORECASE)


def is_tag (token):
  return tag_compiled_pattern.search(token)
print ('Debug: is tag <BODY>', is_tag('<BODY>'))
print ('Debug: is tag something', is_tag('something'))

def is_text (token):
  return not (is_tag(token))

def is_close_tag(tag):
  return tag.startswith('</')

def is_empty_tag(tag):
  return tag.endswith('/>')

def is_instruction_tag(tag):
  return tag.startswith('<!')

def is_open_tag(tag):
  return not (is_close_tag(tag)) and not (is_instruction_tag(tag)) and not (is_empty_tag(tag))

def is_body_tag(tag):
  return body_element_compiled_pattern.search(tag)
print ('Debug: is BODY tag', is_body_tag('<BODY>'))

def get_element_name(tag):
  if is_tag(tag):
    return tag_element_compiled_pattern.search(tag).group(0)
  return None # should never pass by here

def tokenize_SGML (sgml, filter_sgml = True): # return a list of sgml and text tokens
    _current = 0
    #find_tags_result_list = list(re.finditer (tag_pattern, sgml))
    find_tags_result_list = list(tag_compiled_pattern.finditer (sgml))
    #print ('Debug: len(line):',len(sgml),'sgml:',sgml)

  # tokenise into single tag or text sequence
    _tokens = list()
    for tag in find_tags_result_list:
      #print ('Debug:', tag.group(0), tag.start())
      if _current == tag.start(): # document starts with a tag
        if not(filter_sgml):
          token = tag.group(0)
          _tokens.append(token)
        _current = tag.end()
      else:                       # document starts with text
        _tokens.append(sgml[_current:tag.start()]) # we create a token text which covers from the begining of text to the matched tag
        if not(filter_sgml):
          token = tag.group(0)
          _tokens.append(token)
        _current = tag.end()

    # FIXED '<p>[6]  The story of the exploitation of Indians by state and local agencies has been recently summarized by William Brandon:'
    #print ('Debug: find_tags_result>{}<'.format(find_tags_result_list))
    if len(find_tags_result_list) >0: # the last tag is followed by text
      if tag.end() < len(sgml):
        tag = sgml[tag.end():len(sgml)]
        _tokens.append(token)
      #print ('Debug: tag.end()',tag.end())
      else:
        if (tag.end() > len(sgml) or tag.end() < 0) and len(sgml) >0:
          tag = sgml[0:len(sgml)]
          _tokens.append(tag)
    else: # no tags was found in the document
      _tokens.append(sgml) #
    return _tokens #, _token_types


sentence_ending_patterns = ['\p{Ll}\p{Ll}+([\.!\?])\s\p{Lu}\p{Ll}+', '\p{Ll}\p{Ll}+([\.!\?])\s$', '\p{Lu}\p{Ll}\p{Ll}\p{Ll}+([\.!\?])\s\p{Lu}\p{Ll}+', '\p{Lu}\p{Ll}\p{Ll}\p{Ll}+([\.!\?])\s[IA]\s', '(Act|Law)([\.!\?])\s\p{Lu}\p{Ll}\p{Ll}+', '(Act|Law)([\.!\?])\s[IA]\s', '\p{Ll}\p{Ll}+[\'"]*\p{Pe}([\.!\?])\s\p{Lu}\p{Ll}+', '\p{Ll}\p{Ll}+[\'"]*\p{Pe}([\.!\?])\s[IA]\s', '\p{Ps}\d\d+\p{Pe}([\.!\?])\s\p{Lu}\p{Ll}+', '\p{Ps}\d\d+\p{Pe}([\.!\?])\s[IA]\s', '\d\d+([\.!\?])\s\p{Lu}\p{Ll}+', '\d\d+([\.!\?])\s[IA]\s', '\d+[\p{S}\p{P}]\d+([\.!\?])\s\p{Lu}\p{Ll}+', '\d+[\p{S}\p{P}]\d+([\.!\?])\s[IA]\s', '\d\d+([:;])\s\p{Ll}\p{Ll}+', '\d\d+\p{Pe}([:;])\s\p{L}+', '\p{Ps}\d\d+\p{Pe}([:;])\s$', '\d\d+[\p{S}\p{P}]\d\d+([:;])\s', '\p{Ll}\p{Ll}+\p{Pe}([:;])\s$', '\p{L}\p{Ll}+\p{Pe}?([:;])\s\p{L}\p{Ll}+']
# manual revision
# removed '\p{Lu}\p{Ll}\p{Ll}\p{Ll}+[\.!\?]\s\p{Lu}\p{Ll}+', '\p{Lu}\p{Ll}\p{Ll}\p{Ll}+[\.!\?]\s[IA]\s',
# to add ? '\p{Ll}\p{Ll}\p{Ll}+[\.!\?]\s[IA]\s',
# DONE must not tokenize with , "If, for example, <i>Helms I</i>"
#

# https://www.regular-expressions.info/unicode.html
# \p{Ps} or \p{Open_Punctuation}: any kind of opening bracket.
# \p{Pe} or \p{Close_Punctuation}: any kind of closing bracket.
# \p{Pi} or \p{Initial_Punctuation}: any kind of opening quote.
# \p{Pf} or \p{Final_Punctuation}: any kind of closing quote.


# \s+ after the ending separator character
sentence_ending_patterns = [#'\p{Ll}\p{Ll}+[\.!\?]\s\p{Lu}\p{Ll}+',
                            '\w\w+[\.!\?]\s+\p{Lu}\p{Ll}+', #See App. 22a-47a. Helms appealed, ... # ERROR: """cf. Hawaii"""
                            '\w\w+[\.!\?]\[\d+\]\s+\p{Lu}\p{Ll}+', # ...the practice does occur.[16] It would be ...
                            '\w\w+[\.!\?]\[\d+\]\s+\d\d+\s\p{Lu}', # components.[12] 540 F # Appeals.[2] 506 U # make.[3] 474 U
                            '\w\w+[\.!\?]\s+\d\d+\s\p{Lu}', # ... candidate. 361 N.C., at 501-502, 649 S.E.2d, at 371 (case below). """that structure or practice." 478 U.S., at 50, n. 17, 106 S.Ct. 2752. """
                            '\p{Ll}\p{Ll}+[\.!\?]\s+$', # client. # dissent.

                           '\p{Ll}\p{Ll}+[\.!\?]\s+[IA]\s', # pertinent quand textual token élémentaire au sein d'une balise à text tokeniser
                            '(Act|Law)[\.!\?]\s+\p{Lu}\p{Ll}\p{Ll}+',
                            '(Act|Law)[\.!\?]\s+[IA]\s',
                            #'\p{Ll}\p{Ll}+[\'"]*\p{Pe}[\.!\?]\s\p{Lu}\p{Ll}+',
                            #'\p{Ll}\p{Ll}+[\'"]*\p{Pe}[\.!\?]\s[IA]\s',
                            '[\'’"”]*\p{Pe}[\.!\?]\s+\p{Lu}\p{Ll}+', # "780 F.2d 367, 370 (1986) (Helms IV). The court also directed the District..." ; "App. 101a-102a (Directive 801). The District Court's decision was affirmed"
                            '[\'’"”]*\p{Pe}[\.!\?]\s+[IA]\s',
                            '\w\w+[\.!\?]([\'’]\s)?["”]\s+\p{Lu}\p{Ll}+', # """ "... a reasonable attorney's fee as part of the costs." The District Court denied the claim...  """; """ ...his release from prison in 1980." Id., at 10. This statement is... """ ; """...not competitors.' " Ibid., quoting Brown... """
                            '\w\w+[\.!\?]["”]\s+\d\d+\s\p{Lu}', # ... candidate. 361 N.C., at 501-502, 649 S.E.2d, at 371 (case below). """that structure or practice." 478 U.S., at 50, n. 17, 106 S.Ct. 2752. """
                            '\p{Ps}\d\d+\p{Pe}[\.!\?]\s+\p{Lu}\p{Ll}+',
                            '\p{Ps}\d\d+\p{Pe}[\.!\?]\s+[IA]\s',
                            '\d\d+[\.!\?]\s+\p{Lu}\p{Ll}+',
                            '\d\d+[\.!\?]\s+[IA]\s',
                            '\d+[\p{S}\p{P}]\d+[\.!\?]\s+\p{Lu}\p{Ll}+',
                            '\d+[\p{S}\p{P}]\d+[\.!\?]\s+[IA]\s',
                            '\d\d+[:;]\s+["”]?\p{Lu}?\p{Ll}\p{Ll}+',
                            #'\d\d+[:;]\s+\p{Lu}\p{Ll}\p{Ll}+',
                            #'\d\d+\p{Pe}[:;]\s+\p{L}+',
                            '\p{Pe}[:;]\s+["”]?\p{L}+',
                            '\p{Ps}\d\d+\p{Pe}[:;]\s+$',
                            '\d\d+[\p{S}\p{P}]\d\d+[:;]\s+',
                            '\p{Ll}\p{Ll}+\p{Pe}[:;]\s+$',
                            '\p{L}\p{Ll}+\p{Pe}?[:;]\s+["”]?\p{L}\p{Ll}+',
                            '\w\w+\p{Pe}?[\.!\?]\s+§' # ...products (sponsors). §262(l). The Act treats...
                            ] # ,'\n\n' # not possible since there is no punct


# \s+ after the ending separator character
sentence_ending_patterns = [#'\p{Ll}\p{Ll}+[\.!\?]\s\p{Lu}\p{Ll}+',
                            '\w\w+[\.!\?]\s+\p{Lu}\p{Ll}+', #See App. 22a-47a. Helms appealed, ... # ERROR: """cf. Hawaii"""
                            '\w\w+[\.!\?]\[\d+\]\s+\p{Lu}\p{Ll}+', # ...the practice does occur.[16] It would be ...
                            '\w\w+[\.!\?]\[\d+\]\s+\d\d+\s\p{Lu}', # components.[12] 540 F # Appeals.[2] 506 U # make.[3] 474 U
                            '\w\w+[\.!\?]\s+\d\d+\s\p{Lu}', # ... candidate. 361 N.C., at 501-502, 649 S.E.2d, at 371 (case below). """that structure or practice." 478 U.S., at 50, n. 17, 106 S.Ct. 2752. """
                            '\p{L}\p{L}+[\.!\?]\s+$', # client. # dissent.

                           '\p{L}\p{L}+[\.!\?]\s+[IA]\s', # pertinent quand textual token élémentaire au sein d'une balise à text tokeniser
                            '(Act|Law)[\.!\?]\s+\p{Lu}\p{Ll}\p{Ll}+',
                            '(Act|Law)[\.!\?]\s+[IA]\s',
                            #'\p{Ll}\p{Ll}+[\'"]*\p{Pe}[\.!\?]\s\p{Lu}\p{Ll}+',
                            #'\p{Ll}\p{Ll}+[\'"]*\p{Pe}[\.!\?]\s[IA]\s',
                            '[\'’"”]*\p{Pe}[\.!\?]\s+\p{Lu}\p{Ll}+', # "780 F.2d 367, 370 (1986) (Helms IV). The court also directed the District..." ; "App. 101a-102a (Directive 801). The District Court's decision was affirmed"
                            '[\'’"”]*\p{Pe}[\.!\?]\s+[IA]\s',
                            '\w\w+[\.!\?]([\'’]\s)?["”]\s+\p{Lu}\p{Ll}+', # """ "... a reasonable attorney's fee as part of the costs." The District Court denied the claim...  """; """ ...his release from prison in 1980." Id., at 10. This statement is... """ ; """...not competitors.' " Ibid., quoting Brown... """
                            '\w\w+[\.!\?]["”]\s+\d\d+\s\p{Lu}', # ... candidate. 361 N.C., at 501-502, 649 S.E.2d, at 371 (case below). """that structure or practice." 478 U.S., at 50, n. 17, 106 S.Ct. 2752. """
                            '\p{Ps}\d\d+\p{Pe}[\.!\?]\s+\p{Lu}\p{Ll}+',
                            '\p{Ps}\d\d+\p{Pe}[\.!\?]\s+[IA]\s',
                            '\d\d+[\.!\?]\s+\p{Lu}\p{Ll}+',
                            '\d\d+[\.!\?]\s+[IA]\s',
                            '\d+[\p{S}\p{P}]\d+[\.!\?]\s+\p{Lu}\p{Ll}+',
                            '\d+[\p{S}\p{P}]\d+[\.!\?]\s+[IA]\s',
                            '\d\d+[:;]\s+["”]?\p{Lu}?\p{Ll}\p{Ll}+',
                            #'\d\d+[:;]\s+\p{Lu}\p{Ll}\p{Ll}+',
                            #'\d\d+\p{Pe}[:;]\s+\p{L}+',
                            '\p{Pe}[:;]\s+["”]?\p{L}+',
                            '\p{Ps}\d\d+\p{Pe}[:;]\s+$',
                            '\d\d+[\p{S}\p{P}]\d\d+[:;]\s+',
                            '\p{L}\p{L}+\p{Pe}[:;]\s+$',
                            '\p{L}\p{L}+\p{Pe}?[:;]\s+["”]?\p{L}\p{Ll}+',
                            '\w\w+\p{Pe}?[\.!\?]\s+§' # ...products (sponsors). §262(l). The Act treats...
                            ] # ,'\n\n' # not possible since there is no punct

# https://en.wikipedia.org/wiki/List_of_U.S._state_and_territory_abbreviations
us_state_abbrs = ['Wyo.', 'Wis.', 'W. Va.', 'W.Va.', 'D.C.', 'Wash.', 'Va.', 'Vt.', 'V.I.', 'V. I.', 'U.S.', 'U. S.', 'Tex.', 'Tenn.', 'S. Dak.', 'S.Dak.', 'S.D.',  'S. D.','S.C.', 'S. C.','R.I.', 'R. I.', 'P.R.', 'P. R.', 'Pa.', 'Oreg.', 'Ore.', 'Okla.', 'M.P.','M. P.',  'N. Dak.', 'N.Dak.', 'N.D.', 'N. D.','N.C.','N. C.', 'N.Y.', 'N. Y.','N.Mex.', 'N. Mex.', 'N.M.',  'N. M.','N.J.', 'N. J.','N.H.', 'N. H.', 'Nev.', 'Nebr.', 'Neb.', 'Mont.', 'Mo.', 'Miss.', 'Minn.', 'Mich.', 'Mass.', 'Md.', 'La.', 'Ky.', 'Kans.', 'Kan.', 'Iowa', 'Ind.', 'Ill.', 'Ga.', 'Fla.', 'Del.', 'Conn.', 'Colo.', 'Calif.', 'Ark.', 'Ariz.', 'A.S.', 'A. S.', 'Ala.']
# the '.' char in the previous words and ibid. is not a sentence ending separator but such words can occur at the end of a sentence.
# cannot systematically split

# https://en.wikipedia.org/wiki/List_of_legal_abbreviations
legal_abbrs = ['ad.', 'ads.', 'adsm.', 'adj. ', 'Am.', 'Jur.', 'Art.', 'Artt.', 'c.', 'cc.', 'Co.', 'Ct.', 'Cl.', 'Decr.', 'Esq.', 'Fed.', 'Org.', 'N.E.', 'N.W.', 'No.', 'S.E.', 'S.W.', 'p.', 'pp.', 'Relv.', 'Rescr.', 'Resp.', 'sc.', 'viz.', 'Stat.', 'T.C.', 'T.D.', 'v.']

tokens_ending_wi_punct_not_to_split = ['cf.']
tokens_ending_wi_punct_not_to_split.extend(us_state_abbrs)
tokens_ending_wi_punct_not_to_split.extend(legal_abbrs)

def is_token_not_to_split (candidate_context, candidate_punct_position):
# dealing with token ending wi punch which must not be segmented such as cf.
      preceding_token_preceding_punct = re.search('(\w+\.)', candidate_context)
      if preceding_token_preceding_punct:
        preceding_token = preceding_token_preceding_punct.group(1)
        #print ('Debug: >{}<>{}<>{}<>{}<'.format(preceding_token, candidate_context, candidate_punct_position , preceding_token_preceding_punct.end()-1))
        if (candidate_punct_position == preceding_token_preceding_punct.end()-1
          and preceding_token in tokens_ending_wi_punct_not_to_split):
          return True
      return False


import regex
sentence_ending_compiled_patterns = list()

for p in sentence_ending_patterns:
  sentence_ending_compiled_patterns.append(regex.compile(p))

texts = [ #'<A><B>b</B><B>b</B><B><C>c1<d>d</d>c2</C></B></A>',
        "<p>E. Cahn, Our Brother's Keeper 21 (1969), states the same theme:</p>",
        '<p>"<i>That the promises will not be kept.</i></p>',
        '<p>[6] The story of the exploitation of Indians by state</p><p>and local agencies has been recently summarized by William Brandon:</p>',
        '<p>"Termination is truly a word of ill omen to tribal Indians. Its meaning in Indian affairs is the termination of `Federal responsibility,</p>',
        '<p>This construction of the treaty term "down the Arkansas" indicates that at the minimum the boundary of the Choctaws was also the middle of the main channel. Congress was accustomed to using the terms "up" or "down" the river when designating a navigable river as the boundary between States, see, <i>e. g.,</i> Act of March 2, 1819, § 2, 3 Stat. 490 (Alabama); Act of February 20, 1811, § 1, 2 Stat. 641 (Louisiana); and, when it did so, the boundary was set as the middle of the main channel. See <i>Arkansas</i> v. <i>Mississippi,</i> <span class="citation" data-id="99390"><a href="/opinion/99390/arkansas-v-mississippi/"><span class="volume">250</span> <span class="reporter">U.S.</span> <span class="page">39</span> </a></span>(1919); <i>Iowa</i> v. <i>Illinois,</i> <span class="citation" data-id="93467"><a href="/opinion/93467/iowa-v-illinois/"><span class="volume">147</span> <span class="reporter">U.S.</span> <span class="page">1</span> </a></span>(1893).</p>',
        '<p>See Act of June 28, 1898, 30 Stat. 495; Act of July 1, 1902, 32 Stat. 716; Act of April 26, 1906, 34 Stat. 137.</p>',
        '<p>Should it now be held  for purely private purposes? I think not.</p>'
        ]
texts = ['It is common ground that state election-law requirements like the Whole County Provision may be superseded by federal lawfor instance, the one-person, one-vote principle of the Equal Protection Clause of the United States Constitution. See Reynolds v. Sims, 377 U.S. 533, 84 S.Ct. 1362, 12 L.Ed.2d 506 (1964). Here the question is whether § 2 of the Voting Rights Act requires district lines to be drawn that otherwise would violate the Whole County Provision. That, in turn, depends on how the statute is interpreted.']
texts = ['Rules, some state rules including in Missouri, and this Court’s precedents give trial courts some leeway to accept or reject plea agreements, see Fed. Rule Crim. Proc. 11(c)(3); see Mo. Sup. Ct. Rule 24.02(d)(4); Boykin v. Alabama, 395 U. S. 238, 243–244 (1969). It can be assumed']



def segment_text_token (token):
  local_sentence_ends = set() # to avoid distinct patterns to match the same ending
  local_sentence_ending_offsets = list()

  local_sentence_start = 0

  for cp in sentence_ending_compiled_patterns: # run each pattern
    found_sentence_endings = cp.finditer(token)
    # if found_sentence_endings:
    #   print ('Debug: sentence_ending_compiled_pattern',cp,', found_sentence_endings:', found_sentence_endings)
    #   print ('Debug: sentence_ending_compiled_pattern',cp)

    # for all the sentence ending candidates found
    for found_sentence_ending in found_sentence_endings:
      # search the position of the separator in it
      sentence_ending_match = re.search('[\.!?:;]', found_sentence_ending.group(0))

      # dealing with token ending wi punch which must not be segmented such as cf.
      preceding_token_preceding_punct = re.search('(\w+\.)', found_sentence_ending.group(0))
      if not  (is_token_not_to_split (found_sentence_ending.group(0), found_sentence_ending.start())):

      #print ('Debug: pattern:',cp, ';found_sentence_ending:', found_sentence_ending, ';start:',found_sentence_ending.start(),';end:',found_sentence_ending.end(),'; token:',token,'extract:',token[found_sentence_ending.start():found_sentence_ending.end()]) #, 'found_sentence_ending.group(0)', found_sentence_ending.group(0))
        local_sentence_end = found_sentence_ending.start()  + sentence_ending_match.end() #+ previous_found_sentence_ending_start
      #print('Debug: call xml_path 2')
#xpath_sentence_end = xml_path (xpath_edge, xpath_counter_edge, xpath_depth+1)
      #print ('Debug:', found_sentence_ending.start(), sentence_ending_match.start(), local_sentence_end)
        local_sentence_ends.add(local_sentence_end)  # to avoid distinct patterns to match the same ending
        #if local_sentence_ends:
        #    print ('Debug: found_sentence_endings', local_sentence_ends)

  if local_sentence_ends: # has been found, then add intermediary sentences
    #print ('Debug: sorted(list(local_sentence_ends)):',sorted(list(local_sentence_ends)))
    for local_sentence_end in sorted(list(local_sentence_ends)): # need to sort since multiple patterns may have found interlaced sentence endings
      #if has_already_seen_start:
      #print ('Debug: add the sentence ',(sentence_start, local_sentence_end), token[sentence_start:local_sentence_end])
        #print ('Debug: xpath_sentence_start:', xpath_sentence_start,'xpath_sentence_end:', xpath_sentence_end )
      local_sentence_ending_offsets.append((local_sentence_start, local_sentence_end))

#      sentence_ending_xpath_offset.append((xpath_sentence_start, xpath_sentence_end))
      local_sentence_start = local_sentence_end
      #print('Debug: call xml_path 3')
#      xpath_sentence_start = xml_path (xpath_edge, xpath_counter_edge, xpath_depth+1)

      #text_sentence = remove_sgml_and_compress_as_labelstudio(line[sentence_start: local_sentence_end])
      #local_text_sentence_ending_offsets.append((text_sentence_start, text_sentence_start+len(text_sentence)))
      #local_text_sentence.append(text_sentence)
      #text_sentence_start += len(text_sentence)

  # either to capture the last segment or to consider the token as a single segment
  # in case that the end of the last segment does not end the token
  if local_sentence_start +1 != len(token):
    local_sentence_ending_offsets.append((local_sentence_start, len(token)))         #print('Debug: call xml_path 4')
# xpath_sentence_end = xml_path (xpath_edge, xpath_counter_edge, xpath_depth+1)
        #print ('Debug: xpath_sentence_start:', xpath_sentence_start,'xpath_sentence_end:', xpath_sentence_end,'line',line )

  return local_sentence_ending_offsets


def segment_text_into_endstarts (token):
  # given a text return couple of separator offsets i.e. for each separation, the end of the previous and the start of the following
  local_sentence_endstart_offsets = set() # to avoid distinct patterns to match the same ending
  local_sentence_offsets = list()

  local_sentence_start = 0

  for cp in sentence_ending_compiled_patterns: # run each pattern
    found_sentence_endings = cp.finditer(token)
    # if found_sentence_endings:
    #   print ('Debug: sentence_ending_compiled_pattern',cp,', found_sentence_endings:', found_sentence_endings)
    #   print ('Debug: sentence_ending_compiled_pattern',cp)

    # for all the sentence ending candidates found
    for found_sentence_ending in found_sentence_endings:

      # search the position of the separator in it
      sentence_separator_match = re.search('([\.!?:;]["”]?\s*)', found_sentence_ending.group(0))
      #print ('Debug: pattern:',cp, ';found_sentence_ending:', found_sentence_ending, ';start:',found_sentence_ending.start(),';end:',found_sentence_ending.end(),'; token:',token,'extract:',token[found_sentence_ending.start():found_sentence_ending.end()]) #, 'found_sentence_ending.group(0)', found_sentence_ending.group(0))
      #print ('Debug: found_sentence_ending.group(0)=>{}<'.format(found_sentence_ending.group(0)))
      # dealing with token ending wi punch which must not be segmented such as cf.
      #preceding_token_preceding_punct = re.search('(\w+\.)', found_sentence_ending.group(0))
      if not  (is_token_not_to_split (found_sentence_ending.group(0), sentence_separator_match.start())):

        local_sentence_end = found_sentence_ending.start()  + sentence_separator_match.start(1)
        next_sentence_start = found_sentence_ending.start()  + sentence_separator_match.end(1) #+ previous_found_sentence_ending_start
      #print('Debug: call xml_path 2')
#xpath_sentence_end = xml_path (xpath_edge, xpath_counter_edge, xpath_depth+1)
      #print ('Debug:', found_sentence_ending.start(), sentence_separator_match.start(), local_sentence_end)

      # to adjust the sentence ending offset by including the final punctuation
        delta = 1
      # check if the separator contains " ; this is an hack ; depends on the separator pattern search in the search of the position just above
        if '"' in sentence_separator_match.group(1) or '”' in sentence_separator_match.group(1):
          delta = 2

        local_sentence_endstart_offsets.add((local_sentence_end+delta, next_sentence_start))  # to avoid distinct patterns to match the same ending # +1 to include the separator character
        #if local_sentence_endstart_offsets:
        #    print ('Debug: found_sentence_endings', local_sentence_endstart_offsets)

  if local_sentence_endstart_offsets: # has been found, then add intermediary sentences
    #print ('Debug: sorted(list(local_sentence_endstart_offsets)):',sorted(list(local_sentence_endstart_offsets)))
    for local_sentence_end, next_sentence_start in sorted(list(local_sentence_endstart_offsets)): # need to sort since multiple patterns may have found interlaced sentence endings
      #if has_already_seen_start:
      #print ('Debug: add the sentence ',(sentence_start, local_sentence_end), token[sentence_start:local_sentence_end])
        #print ('Debug: xpath_sentence_start:', xpath_sentence_start,'xpath_sentence_end:', xpath_sentence_end )
      local_sentence_offsets.append((local_sentence_start, local_sentence_end))

#      sentence_ending_xpath_offset.append((xpath_sentence_start, xpath_sentence_end))
      local_sentence_start = next_sentence_start
      #print('Debug: call xml_path 3')
#      xpath_sentence_start = xml_path (xpath_edge, xpath_counter_edge, xpath_depth+1)

      #text_sentence = remove_sgml_and_compress_as_labelstudio(line[sentence_start: local_sentence_end])
      #local_text_sentence_ending_offsets.append((text_sentence_start, text_sentence_start+len(text_sentence)))
      #local_text_sentence.append(text_sentence)
      #text_sentence_start += len(text_sentence)

  # either to capture the last segment or to consider the token as a single segment
  # in case that the end of the last segment does not end the token
  if local_sentence_start +1 != len(token):
    local_sentence_offsets.append((local_sentence_start, len(token)))         #print('Debug: call xml_path 4')
# xpath_sentence_end = xml_path (xpath_edge, xpath_counter_edge, xpath_depth+1)
        #print ('Debug: xpath_sentence_start:', xpath_sentence_start,'xpath_sentence_end:', xpath_sentence_end,'line',line )

  return local_sentence_offsets



import re

amp_re = '&amp;'
amp_compiled_re = re.compile(amp_re)

def compute_delta_amp_entity_occurrences (text):
  # LS count the len of the entity character & and not the &amp;
  # this code is a fix
  # count how many amp entities
  # compute the length to remove
  result = amp_compiled_re.findall(text)
  return -1*len(result) * len('&amp;')+len(result)

def determine_global_local_offsets_plus_text (global_text, local_text, local_sentence_offsets, xpath_local_start, xpath_local_end, tokenize_as_single_text_unit = False):
  # 240205 this version considers local_text as a single text unit and not separated text tokens
  # 240206 this version should work with not considering sup, pagination... and consider separators offset as a pair of point the ending of the previous one and the beginning of the new one ; see segment_text_into_endstarts
  # TODO remove local_sentence_offsets parameter which seems not to be used

  local_text = [''.join(local_text)]
  global_start_of_local_text =  len(''.join(global_text)) - len(local_text[0])

  annotations = list()
  local_text_token_start = 0
  local_offset_start = local_text_token_start
  # local_text covers the most embedded text
  #print ('Debug: local_text>{}<'.format(local_text))

  if not (tokenize_as_single_text_unit):
    for local_text_token in local_text:
      # segment each part of the local_text
      # return a list of pair of offsets
      local_sentence_offsets = segment_text_into_endstarts (local_text_token)
      #print ('Debug: len(local_sentence_offsets)>{}< local_sentence_offsets>{}<'.format(len(local_sentence_offsets), local_sentence_offsets))
      if len(local_sentence_offsets)>1:
        for local_sentence_offset in local_sentence_offsets:
          local_offset_start = local_text_token_start + local_sentence_offset[0]
          local_offset_end = local_text_token_start + local_sentence_offset[1]  #+ compute_delta_amp_entity_occurrences (local_text_token) # Commented thanks to the ugly "hack" which solves entities in parse...
          global_offset_start = global_start_of_local_text + local_offset_start
          global_offset_end = global_start_of_local_text + local_offset_end
          annotations.append ((local_offset_start, local_offset_end, global_offset_start, global_offset_end, local_text_token[local_sentence_offset[0]:local_sentence_offset[1]], xpath_local_start, xpath_local_end))

      local_text_token_start += len (local_text_token) # HERE TOO ? + compute_delta_amp_entity_occurrences (local_text_token)

  if tokenize_as_single_text_unit or len(annotations) == 0:
    local_offset_end = len(local_text[0])   # HERE TOO ? + compute_delta_amp_entity_occurrences (local_text_token) # local_text_token_start + local_sentence_ending_offset[1]
    global_offset_start = global_start_of_local_text + local_offset_start
    global_offset_end = global_start_of_local_text + local_offset_end
    #print ('Debug:',local_offset_start, local_offset_end, global_offset_start, global_offset_end, ''.join(local_text)[local_offset_start:local_offset_end], xpath_local_start, xpath_local_end)
    annotations.append ((local_offset_start, local_offset_end, global_offset_start, global_offset_end, local_text[0][local_offset_start:local_offset_end], xpath_local_start, xpath_local_end))

  return annotations


def xml_path (xpath_edge, xpath_counter_edge, xpath_depth):
  xpath_steps = list()
  #print ('Debug: GENERATE XML PATH')
  #print ('Debug: xpath_edge',xpath_edge)
  #print ('Debug: xpath_counter_edge', xpath_counter_edge)
  #print ('Debug: xpath_depth', xpath_depth)
  for i in range(0,xpath_depth +1):
    #if xpath_edge[i] in xpath_counter_edge[i]:
      #print ('Debug: xpath step',i, ''.join(['/', xpath_edge[i], '[', str(xpath_counter_edge[i][xpath_edge[i]]), ']']))
      xpath_steps.append(''.join(['/', xpath_edge[i], '[', str(xpath_counter_edge[i][xpath_edge[i]]), ']']))
    #else:
    #  print ('Debug: partial xpath step',i, ''.join(['/', xpath_edge[i], '[]']))
    #  xpath_steps.append(''.join(['/', xpath_edge[i], '[]']))
  #print ('Debug: xpath_steps', xpath_steps)
  if not(xpath_steps[-1].startswith('/text')): # FIXME: brief patch...
    xpath_steps.append('/text()[1]')
  return ''.join(xpath_steps)

def update_xpath_variables_wi_open_tag (element, xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge):
          xpath_depth += 1
          if xpath_depth > xpath_depth_max: xpath_depth_max = xpath_depth

          for j in range(xpath_depth+1, xpath_depth_max+1):
          #  print ('Debug: clean', j, xpath_edge[j])
            xpath_counter_edge[j] = dict()
            xpath_edge[j] = ''

          if not (element in xpath_counter_edge[xpath_depth]):
            xpath_counter_edge[xpath_depth][element] = 1
            xpath_edge[xpath_depth] = element
          else:
            xpath_counter_edge[xpath_depth][element] += 1
            xpath_edge[xpath_depth] = element
          return           xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge

def update_xpath_variables_wi_close_tag (xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge):
            for j in range(xpath_depth+1, xpath_depth_max):
              xpath_counter_edge[j] = dict()
              xpath_edge[j] = ''
            xpath_depth -= 1
            return xpath_depth, xpath_counter_edge, xpath_edge

def update_xpath_variables_wi_text (xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge):
        #print ('Debug: BEFORE update_xpath_variables_wi_text')
        #print ('Debug: xpath_edge',xpath_edge)
        #print ('Debug: xpath_counter_edge', xpath_counter_edge)
        #print ('Debug: xpath_depth', xpath_depth)

        xpath_depth += 1
        if xpath_depth > xpath_depth_max: xpath_depth_max = xpath_depth
        #print ('Debug: xpath_depth_max', xpath_depth_max)
        if not ('text()' in xpath_counter_edge[xpath_depth]): #[xpath_depth] != element:
          xpath_counter_edge[xpath_depth]['text()'] = 1
          xpath_edge[xpath_depth] = 'text()'
        else:
          xpath_counter_edge[xpath_depth]['text()'] += 1
          xpath_edge[xpath_depth] = 'text()'

        #print ('Debug: AFTER update_xpath_variables_wi_text 1')
        #print ('Debug: xpath_edge',xpath_edge)
        #print ('Debug: xpath_counter_edge', xpath_counter_edge)
        #print ('Debug: xpath_depth', xpath_depth)
        #print ('Debug: xpath_depth_max', xpath_depth_max)

        for j in range(xpath_depth+1, xpath_depth_max+1):
        #  print ('Debug: clean', j, xpath_edge[j])
          xpath_counter_edge[j] = dict()
          xpath_edge[j] = ''
        xpath_depth -= 1
        #xpath_depth, xpath_counter_edge, xpath_edge = update_xpath_variables_wi_close_tag (xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge)

        #print ('Debug: AFTER update_xpath_variables_wi_text AND AFTER clean')
        #print ('Debug: xpath_edge',xpath_edge)
        #print ('Debug: xpath_counter_edge', xpath_counter_edge)
        #print ('Debug: xpath_depth', xpath_depth)
        #print ('Debug: xpath_depth_max', xpath_depth_max)

        return xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge

import random

def generate_LS_id():
  # "-4HPZ2H7dG"
  # https://stackoverflow.com/questions/2511222/efficiently-generate-a-16-character-alphanumeric-string
  return ''.join(random.choice('0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz-') for i in range(10))

print ('Debug: generate_LS_id():', generate_LS_id())
print ('Debug: generate_LS_id():', generate_LS_id())

import math
import secrets
def random_alphanum(length: int) -> str:
  # https://stackoverflow.com/questions/2511222/efficiently-generate-a-16-character-alphanumeric-string
    if length == 0:
        return ''
    elif length < 0:
        raise ValueError('negative argument not allowed')
    else:
        text = secrets.token_hex(nbytes=math.ceil(length / 2))
        is_length_even = length % 2 == 0
        return text if is_length_even else text[1:]
print ('Debug: random_alphanum(12):', random_alphanum(12))

def generate_LS_unique_id():
#	"6f93b7a7-72c5-4bd3-a8b8-8fd564d91456"
  unique_id = list()
  unique_id.append(random_alphanum(8))
  unique_id.append('-')
  unique_id.append(random_alphanum(4))
  unique_id.append('-')
  unique_id.append(random_alphanum(4))
  unique_id.append('-')
  unique_id.append(random_alphanum(4))
  unique_id.append('-')
  unique_id.append(random_alphanum(12))
  return ''.join(unique_id)

print ('Debug: generate_LS_unique_id():', generate_LS_unique_id())
print ('Debug: generate_LS_unique_id():', generate_LS_unique_id())


def new_annotation (xpath_start, xpath_end,
                    startOffset, endOffset,
                    globalOffsets_start, globalOffsets_end,
                    text, annotation_labels,
                    annotation_type = "labels"
                    ):
  #_id = generate_LS_id() #hash(''.join([text, str(globalOffsets_start), ''.join(annotation_labels)]))
  return {
            "value": {
              "start": xpath_start,
              "startOffset": startOffset,
              "end": xpath_end,
              "endOffset": endOffset,
              "globalOffsets": {
                "start": globalOffsets_start,
                "end": globalOffsets_end
              },
              "text": text,
              "labels": annotation_labels
            },
            "id": generate_LS_id(), # "68rYy_SMpL",
            "from_name": "label",
            "to_name": "text",
            "type": annotation_type,
            "origin": "manual"
          }



def new_task (annotation_id, annotator_id, result):
  #_id = generate_LS_id() #hash(''.join([text, str(globalOffsets_start), ''.join(annotation_labels)]))
  return {
        "id": annotation_id,
        "completed_by": annotator_id,
        "result": result,
        "was_cancelled": False,
        "ground_truth": False,
        "created_at": "2024-01-24T23:13:53.738091Z",
        "updated_at": "2024-01-24T23:13:53.738122Z",
        "draft_created_at": None,
        "lead_time": 30.378,
        "prediction": {},
        "result_count": 0,
        "unique_id": generate_LS_unique_id(), # unique_id, #"63d34cdd-1ddb-4b91-9673-bb176c753655",
        "import_id": None,
        "last_action": None,
        "task": 1,
        "project": 1,
        "updated_by": annotator_id,
        "parent_prediction": None,
        "parent_annotation": None,
        "last_created_by": None
      }

def new_annotated_document (document_id, file_upload, task, text, project_id = 1, annotator_id = 1, is_annotations = True):

  if is_annotations:
    annotations = [task]
    predictions = []
  else:
    annotations = []
    predictions = [task]

    #score = dict()
    #score["score"] = 0.95
    #predictions.append(score) # the score is fake

  return {
      'id' : document_id,
      'annotations' : annotations,
      'file_upload': file_upload,
      'file_name' : file_upload, # field created by NH https://labelstud.io/guide/tasks#Basic-Label-Studio-JSON-format
      'drafts' : [],
      "predictions": predictions,
      "data": {"text": text },
      'meta' : {},
      'created_at' : "2024-01-24T23:27:40.198798Z",
      'updated_at' : "2024-01-24T23:28:29.871312Z",
      'inner_id' : document_id,
      'total_annotations' : 1, #len(task),
      'cancelled_annotations': 0,
      'total_predictions': 0,
      'comment_count': 0,
      'unresolved_comment_count': 0,
      'last_comment_updated_at': None,
      'project' : project_id,
      'updated_by' : annotator_id,
      'comment_authors' : []
  }

import re
import json
import html

def parse (html_content):
  """
  return the LS interpretation of the given html_content and a list of sentence annotation metadata
  """
  #html_content = "<html><head></head><body><div><p>hello</p><p>world !</p><p>hello<b>world</b> !</p></div></body></html>" # example1
  #html_content = "<html>\n<head></head><body><div>\n<p>hello</p><p>world !</p><p>hello<b>world</b> !</p></div></body></html>"# example2
  #html_content = '<html>\n<head></head><body>\n<p class="bottom">\n<span class="meta-data-header">Author:</span>\n<span class="meta-data-value">\n<a href="/person/288/hugo-lafayette-black/">Hugo Lafayette Black</a>\n</span>\n</p>\n<br/>\n<div class="tab-content">\n<div id="010combined1">\n<div class="v-offset-above-2" id="lawbox-text">\n<div id="opinion-content">\n<div class="serif-text"><div>\n<center><b><span class="citation no-link"><span class="volume">344</span> <span class="reporter">U.S.</span> <span class="page">157</span> </span>(1952)</b></center><center><h1>LLOYD A. FRY ROOFING CO.<br/>\nv.<br/>\nWOOD ET AL., MEMBERS OF THE ARKANSAS PUBLIC SERVICE COMMISSION.</h1></center></div></div></div></div></div></div>\n</body></html>'
  #html_content = '<html><head></head><body><p class="bottom"><span class="meta-data-header">Author:</span><span class="meta-data-value">Hugo Lafayette Black</span></p><center>b</center><center><h1>LLOYD A. FRY ROOFING CO.<br/>v.<br/>WOOD ET AL., MEMBERS OF THE ARKANSAS PUBLIC SERVICE COMMISSION.</h1></center></body>'
  #html_content = '<html><head></head><body><p>[5]  In <i>Columbia Terminals Co.</i> v. <i>Lambert,</i> <span class="citation" data-id="1378436"><a href="/opinion/1378436/columbia-terminals-co-v-lambert/"><span class="volume">30</span> <span class="reporter">F. Supp.</span> <span class="page">28</span></a></span>, the District Court upheld a Missouri statute reading: "It is hereby declared unlawful for any motor carrier . . . to use any of the public highways of this state for the transportation of persons or property, or both, in interstate commerce without first having obtained from the commission a permit so to do. . . ." <i>Buck</i> v. <i>Kuykendall,</i> <span class="citation" data-id="100587"><a href="/opinion/100587/buck-v-kuykendall/"><span class="volume">267</span> <span class="reporter">U.S.</span> <span class="page">307</span></a></span>, was held not to require the statute\'s invalidation, since Missouri had not refused to grant a permit on the ground that the state had power to say what interstate commerce would benefit the state and what would not. Agreeing with this constitutional holding, we ordered the complaint dismissed. <span class="citation no-link"><span class="volume">309</span> <span class="reporter">U.S.</span> <span class="page">620</span></span>. See also <i>Eichholz</i> v. <i>Public Service Comm\'n,</i> <span class="citation" data-id="103154"><a href="/opinion/103154/eichholz-v-public-serv-commn-of-mo/"><span class="volume">306</span> <span class="reporter">U.S.</span> <span class="page">268</span></a></span>, 273-274; <i>H. P. Welch Co.</i> v. <i>New Hampshire,</i> <span class="citation" data-id="103140"><a href="/opinion/103140/hp-welch-co-v-new-hampshire/"><span class="volume">306</span> <span class="reporter">U.S.</span> <span class="page">79</span></a></span>, 84, 85; <i>Maurer</i> v. <i>Hamilton,</i> <span class="citation" data-id="103338"><a href="/opinion/103338/maurer-v-hamilton/"><span class="volume">309</span> <span class="reporter">U.S.</span> <span class="page">598</span></a></span>, affirming <span class="citation" data-id="3851725"><a href="/opinion/4092343/maurer-v-boardman/"><span class="volume">336</span> <span class="reporter">Pa.</span> <span class="page">17</span></a></span>, <span class="citation" data-id="3851725"><a href="/opinion/4092343/maurer-v-boardman/"><span class="volume">7</span> <span class="reporter">A.2d</span> <span class="page">466</span></a></span>; <i>McDonald</i> v. <i>Thompson,</i> <span class="citation" data-id="103102"><a href="/opinion/103102/mcdonald-v-thompson/"><span class="volume">305</span> <span class="reporter">U.S.</span> <span class="page">263</span></a></span>, affirming <span class="citation" data-id="1563136"><a href="/opinion/1563136/thompson-v-mcdonald/"><span class="volume">95</span> <span class="reporter">F.2d</span> <span class="page">937</span></a></span>; <i>South Carolina State Highway Dept.</i> v. <i>Barnwell Bros., Inc.,</i> <span class="citation" data-id="102963"><a href="/opinion/102963/sc-hwy-dept-v-barnwell-bros/"><span class="volume">303</span> <span class="reporter">U.S.</span> <span class="page">177</span></a></span>. Cf. <i>Buck</i> v. <i>Kuykendall,</i> <span class="citation" data-id="100587"><a href="/opinion/100587/buck-v-kuykendall/"><span class="volume">267</span> <span class="reporter">U.S.</span> <span class="page">307</span></a></span>, and <i>Hood &amp; Sons, Inc.</i> v. <i>Du Mond,</i> <span class="citation" data-id="104652"><a href="/opinion/104652/hp-hood-sons-inc-v-du-mond/"><span class="volume">336</span> <span class="reporter">U.S.</span> <span class="page">525</span></a></span>, 538.</p></body>'
  #html_content = '<html><head></head><body><p>[5]  In <i>Columbia Terminals Co.</i> v. <i>Lambert,</i> <span class="citation" data-id="1378436"><a href="/opinion/1378436/columbia-terminals-co-v-lambert/"><span class="volume">30</span> <span class="reporter">F. Supp.</span> <span class="page">28</span></a></span>, the District Court upheld a Missouri statute reading: "It is hereby declared unlawful for any motor carrier . . . to use any of the public highways of this state for the transportation of persons or property, or both, in interstate commerce without first having obtained from the commission a permit so to do. . . ." <i>Buck</i> v. <i>Kuykendall,</i> <span class="citation" data-id="100587"><a href="/opinion/100587/buck-v-kuykendall/"><span class="volume">267</span> <span class="reporter">U.S.</span> <span class="page">307</span></a></span>, was held not to require the statute\'s invalidation, since Missouri had not refused to grant a permit on the ground that the state had power to say what interstate commerce would benefit the state and what would not. Agreeing with this constitutional holding, we ordered the complaint dismissed. <span class="citation no-link"><span class="volume">309</span> <span class="reporter">U.S.</span> <span class="page">620</span></span>. See also <i>Eichholz</i> v. <i>Public Service Comm\'n,</i> <span class="citation" data-id="103154"><a href="/opinion/103154/eichholz-v-public-serv-commn-of-mo/"><span class="volume">306</span> <span class="reporter">U.S.</span> <span class="page">268</span></a></span>, 273-274; <i>H. P. Welch Co.</i> v. <i>New Hampshire,</i> <span class="citation" data-id="103140"><a href="/opinion/103140/hp-welch-co-v-new-hampshire/"><span class="volume">306</span> <span class="reporter">U.S.</span> <span class="page">79</span></a></span>, 84, 85; <i>Maurer</i> v. <i>Hamilton,</i> <span class="citation" data-id="103338"><a href="/opinion/103338/maurer-v-hamilton/"><span class="volume">309</span> <span class="reporter">U.S.</span> <span class="page">598</span></a></span>, affirming <span class="citation" data-id="3851725"><a href="/opinion/4092343/maurer-v-boardman/"><span class="volume">336</span> <span class="reporter">Pa.</span> <span class="page">17</span></a></span>, <span class="citation" data-id="3851725"><a href="/opinion/4092343/maurer-v-boardman/"><span class="volume">7</span> <span class="reporter">A.2d</span> <span class="page">466</span></a></span>; <i>McDonald</i> v. <i>Thompson,</i> <span class="citation" data-id="103102"><a href="/opinion/103102/mcdonald-v-thompson/"><span class="volume">305</span> <span class="reporter">U.S.</span> <span class="page">263</span></a></span>, affirming <span class="citation" data-id="1563136"><a href="/opinion/1563136/thompson-v-mcdonald/"><span class="volume">95</span> <span class="reporter">F.2d</span> <span class="page">937</span></a></span>; <i>South Carolina State Highway Dept.</i> v. <i>Barnwell Bros., Inc.,</i> <span class="citation" data-id="102963"><a href="/opinion/102963/sc-hwy-dept-v-barnwell-bros/"><span class="volume">303</span> <span class="reporter">U.S.</span> <span class="page">177</span></a></span>. Cf. <i>Buck</i> v. <i>Kuykendall,</i> <span class="citation" data-id="100587"><a href="/opinion/100587/buck-v-kuykendall/"><span class="volume">267</span> <span class="reporter">U.S.</span> <span class="page">307</span></a></span>, and <i>Hood &amp; Sons, Inc.</i> v. <i>Du Mond,</i> <span class="citation" data-id="104652"><a href="/opinion/104652/hp-hood-sons-inc-v-du-mond/"><span class="volume">336</span> <span class="reporter">U.S.</span> <span class="page">525</span></a></span>, 538.</p></body>'
  #print('Debug:', html_content[:100])

  html_elements_to_text_tokenize = ['p', 'blockquote', ] # 'blockquote', 'center', 'h1',  'li', 'ul'] # 'div',  'h1', 'h2', 'h3', 'h4', 'h5','pre',
  html_elements_to_tokenize_as_single_text_unit = ['blockquote']
  # compris dans des balises <div id="opinion-content"> <div class="plaintext"><pre class="inline"> https://www.courtlistener.com/opinion/3216495/fisher-v-university-of-tex-at-austin/

  # TODO
  # should be visually kept... but not considered when searching tokenization patterns
  # """<sup>[1]</sup>""", """<span class="page">46</span>"""  """ <span class="star-pagination">*157</span>""" <span class=\"star-pagination\">
  start_open_tags_to_make_disapear = ['<sup>', '<span class="page">', '<span class="star-pagination">']

  # Tokenize
  tokens = tokenize_SGML(html_content.strip(), False)  # de nombreux fichiers HTML commencent par une ligne vide dans notre corpus bug ?

  #print ('Debug: tokens:',tokens[:100])

  # Managing XPath computation
  xpath_counter_edge = [dict() for i in range(1000)]  # listes de compteurs d'étiquettes (map) à une profondeur donnée (rang dans la table) # arbitrary depth length
  xpath_edge = ['' for i in range(1000)]  # dernière étiquette rencontrée à un niveau de profondeur donnée # arbitrary depth length
  xpath_depth = -1 # depth index in the SGML structure
  xpath_depth_max = -1 # depth index max

  #
  annotations = list()

  local_text = list() # corresponding to a text embedded in a pair of open/close tag of a given element
  global_text = list()
  stack_of_embedded_elements_to_segment = list() # when several elements to segment, segment the most embedding one, and deal with recursive embedded identical elements
  stack_of_embedded_tags = list() # to handle some token tags not to process in tokenization such as """<sup>[1]</sup>""", """<span class="page">46</span>"""  """ <span class="star-pagination">*157</span>"""

  has_xpath_local_start = False
  in_body_tag = False
  in_a_tag_to_make_disapear = False # """<sup>[1]</sup>""", """<span class="page">46</span>"""  """ <span class="star-pagination">*157</span>"""

  for token_index in range (0, len(tokens)):
    token = tokens[token_index] # current token (tag or text)

    if is_tag (token):
      element = get_element_name(token)

      #print ('Debug: is_tag:', element)

      if is_open_tag(token):
        #print ('Debug: is_start_tag >{}<'.format(token))

        #
        if is_body_tag(token): in_body_tag = True

        #
        #BYPASS# xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge = update_xpath_variables_wi_open_tag (element, xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge)

        # if tag is a tag whose content has to be segmented
        if element in html_elements_to_text_tokenize:
          #print ('Debug: is_element_to_segment')

          local_text = list()

          stack_of_embedded_elements_to_segment.append(element)
          #print ('Debug: append element>{}< stack_of_embedded_elements_to_segment>{}<'.format(element,stack_of_embedded_elements_to_segment))
          #stack_of_embedded_tags.append(token)
        if token in start_open_tags_to_make_disapear:
          #print ('Debug: start_open_tags_to_make_disapear:>{}<'.format(token))
          in_a_tag_to_make_disapear = True

      elif is_close_tag (token):               # is_close_tag
        #print ('Debug: is_close_tag >{}<'.format(token))

        # if tag is a tag whose content has to be segmented
        if element in html_elements_to_text_tokenize:
          #print ('Debug: is_element_to_segment')

          #print ('Debug: pop element>{}< stack_of_embedded_elements_to_segment>{}<'.format(element,stack_of_embedded_elements_to_segment))
          stack_of_embedded_elements_to_segment.pop(len(stack_of_embedded_elements_to_segment)-1)
          #last_open_tag_token = stack_of_embedded_tags[len(stack_of_embedded_elements_to_segment)]
          #stack_of_embedded_tags.pop(len(stack_of_embedded_elements_to_segment)-1)

          # case of empty local_text with <p></p> which arises an error in xml_path: FIXED with a condition
          #print ('Debug: before xml_path, local_text>{}<:'.format(local_text))

          #BYPASS#if len(local_text) != 0:
          #BYPASS#  xpath_local_end = xml_path (xpath_edge, xpath_counter_edge, xpath_depth+1)

          # process the text segmentation when the most embedding tag is closed
          if len(stack_of_embedded_elements_to_segment) == 0:
              tokenize_as_single_text_unit = False
              if element in html_elements_to_tokenize_as_single_text_unit:
                tokenize_as_single_text_unit = True

              local_sentence_ending_offsets = None
              #BYPASS# annotations.extend(determine_global_local_offsets_plus_text (global_text, local_text, local_sentence_ending_offsets, xpath_local_start, xpath_local_end, tokenize_as_single_text_unit))
              has_xpath_local_start = False

        in_a_tag_to_make_disapear = False

        #
        #BYPASS# xpath_depth, xpath_counter_edge, xpath_edge = update_xpath_variables_wi_close_tag (xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge)

      elif is_empty_tag (token):
        if re.match ('br', element, re.IGNORECASE): # LS change the <br/> tags into '\n'...
          #print ('Debug: is_empty_tag>{}<'.format(token))

          local_text.append('\n')
          if in_body_tag: # LS starts to consider global offset from the body tag
            global_text.append('\n')
        # else if is instruction tag...

    else:                    # is_text
      #print ('Debug: is_text>{}<'.format(token))

      # FIXME the following line is an ugly hack to fix the fact that LS takes the len of interpreted entities and not the entity codes
      # should not do that ! because &lt; and &gt; may be changed too and create bugs in interpreting HTML in LS
      # Note: LS annotation seems to keep the unescaped form but not the data field (global)
      token = html.unescape(token)

      if in_a_tag_to_make_disapear:
        token = ' '*len(token)

      #
      #BYPASS# xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge = update_xpath_variables_wi_text (xpath_depth, xpath_depth_max, xpath_counter_edge, xpath_edge)

      # if in a tag to segment
      if len(stack_of_embedded_elements_to_segment) >0:
        if not(has_xpath_local_start):
          #BYPASS# xpath_local_start = xml_path (xpath_edge, xpath_counter_edge, xpath_depth+1)
          has_xpath_local_start = True
        local_text.append(token)

      if in_body_tag: # LS starts to consider global offset from the body tag
        global_text.append(token)
    #print()
  #print ('Debug: annotations>{}<'.format(annotations[:100]))
  #print ('Debug: tokens:',tokens[:100])
  #print ('Debug: global_text>{}<'.format(''.join(global_text)))
  #
  #with open('global_text.txt', 'w') as f:
  #  f.write(''.join(global_text))

  return global_text, annotations


from bs4 import BeautifulSoup
import re

def is_empty_line(line):
  return len(line.strip()) == 0
  #return re.match('^$',re.sub('\s*', '', line))

def ws_at_the_begining_len (line):
  ws_at_the_begining_found = re.search('^(\s*)', line)
  if ws_at_the_begining_found:
    return len(ws_at_the_begining_found.group(1))
  return 0

def total_not_ws_char(line):
  total_not_ws_char_found = re.findall('[^\s]',line)
  if total_not_ws_char_found:
    return len(total_not_ws_char_found)
  return 0

def trim_header_footnotes (lines):
  trimed_lines = list()
  footnotes = list()
  header_left = list()
  header_right = list()
  page_index = 1
  is_footnote = 0
  i = 0

  indent = dict()
  average_line_len = 0

  while i < len(lines):
    line = lines[i]

    current_indent = ws_at_the_begining_len(line)
    if current_indent in indent: indent[current_indent] += 1
    else: indent[current_indent] = 1
    average_line_len += len(line)

    # begining of a header
    if re.search('^\u000C', line, re.UNICODE):

      # if previous lines were dedicated for footnotes
      is_footnote = 0

      # processing the header
      page_index += 1
      if page_index %2 == 0 and len(header_left) == 0:
        for j in range (i, i+4):
          if j < len(lines):
            header_left.append(lines[j])
      elif page_index %2 != 0 and len(header_right) == 0:
        for j in range (i, i+4):
          if j < len(lines):
            header_right.append(lines[j])

      # we skip the header lines
      for j in range (i, i+4):
        if j < len(lines):
          i = i+1

    # begining of a footnote
    elif re.search('^——————$', line):
      # we assume the footnote will be stopped by encountering a header
      is_footnote = 1
      #print ('Debug:', lines[i])

      # test if the previous line was empty, if so, removed it
      j = i-1
      #print ('Debug: current lines[-1]>{}<lines[j]>{}<len(lines)>{}<'.format(lines[-1],lines[j], len(lines)))
      while j>=0 and len(lines[j].strip()) == 0 :
        #print ('Debug: removing lines[-1]>{}<><'.format(lines[-1]))
        #del(lines[-1])
        trimed_lines.pop()
        j -= 1
        #print ('Debug: next lines[j]>{}<len(lines)>{}<'.format(lines[j], len(lines)))

      #print ('Debug: now lines[-1]{}<len(lines)>{}<'.format(trimed_lines[-1], len(lines)))
      #print ('Debug:------------------')
      #print ('Debug:',lines[i])
      # we skip the current line
      i += 1
    else:
      if is_footnote:
        if len(line.strip()) !=0:
          footnotes.append(line)
      else:
        trimed_lines.append(line)
      i += 1
  return trimed_lines, header_left, header_right, footnotes, indent, average_line_len/len(lines)



roman_numbers = ['I', 'II', 'III', 'IV', 'V', 'V', 'VI', 'VIII', 'IX', 'X', 'XI', 'XII', 'XIII', 'XIV', 'XV', 'XVI1', 'XVII', 'XVIII', 'XIX', 'XX']


def is_roman_number (line):
  return line.strip() in roman_numbers

import re

def is_part_separator(line):
  return re.search ('*     *     *', line)

def extract_bullet_point (line):
  result = re.search('^\s*(\w+)$', line)
  if not (result):
      result = re.search('^\s*(\w+)\.', line)
  if result:
    point = result.group(1)
    if point in roman_numbers:
      return 'roman'
    elif re.match('^[A-Z]$', point):
      return 'letter'
    elif re.match('^\d+$', point):
      return 'figure'
  return None

def segment_into_parag_and_heading_on_layout(lines):

    # parse lines to determine whether to insert paragraphs
    i = 0
    new_html_content = list()
    new_html_content.append('<p>')
    paragraph_open = 1

    #section_roman_title_index = 0
    #subsect_alpha_title_index = 0

    max_indent = 4
    average_total_not_ws_char=47
    average_line_len = 45
    still_line_empty = 0
    debug = False
    while i < len(lines):
      line = lines[i]

      # is empty line
      if is_empty_line(line):
        if debug: new_html_content.append('Debug: ---------------------------is empty line')
        if paragraph_open: # and not (still_line_empty):
          new_html_content.append('</p>')
          #new_html_content.append('<p>')
          paragraph_open = 0
          still_line_empty = 1
        new_html_content.append(line)
        i += 1

      # is indentation
      elif (i+1<=len(lines) and i>0
          and (ws_at_the_begining_len(line) != ws_at_the_begining_len(lines[i-1]))
          and (ws_at_the_begining_len(line) != ws_at_the_begining_len(lines[i+1]))
            #and (ws_at_the_begining_len(line) <= max_indent)
          and len(line) >= average_line_len*80/100): # on average total_not_ws_char=47
        if debug: new_html_content.append('Debug: --------------------------is indentation')
        if paragraph_open: # and not (still_line_empty):
           new_html_content.append('</p>')
        new_html_content.append('<p>')
        new_html_content.append(line)
        paragraph_open = 1
        still_line_empty = 0
        i += 1

      # is title and may be a section headings
      elif (total_not_ws_char(line) >0
            and ws_at_the_begining_len(line) > max_indent
            and total_not_ws_char(line) < average_total_not_ws_char - max_indent):
        if debug: new_html_content.append('Debug: --------------------------------is center or heading wi point>{}<'.format(extract_bullet_point(line)))
        point = extract_bullet_point(line)
        if paragraph_open: # and not (still_line_empty):
          new_html_content.append('</p>')
        if point == 'roman':
            new_html_content.append('<h2>')
        elif point == 'letter':
          new_html_content.append('<h3>')
        elif point == 'figure':
          new_html_content.append('<h4>')
        else:
          new_html_content.append('<p>')
          #new_html_content.append('<center>')
        #print ('Debug:>{}<'.format(line))
        new_html_content.append(line)
        i += 1
        if point == 'roman':
          new_html_content.append('</h2>')
        elif point == 'letter':
          new_html_content.append('</h3>')
        elif point == 'figure':
          new_html_content.append('</h4>')
        else:
          #new_html_content.append('</center>')
          new_html_content.append('</p>')
        #new_html_content.append('<p>')
        paragraph_open = 0
        still_line_empty = 0

      else:
        #if debug: new_html_content.append('Debug: --------------------------------is other')
        if not (paragraph_open):
          new_html_content.append('<p>')
          paragraph_open = 1
        new_html_content.append(line)
        still_line_empty = 0
        i += 1
    new_html_content.append('</p>')

    return new_html_content


def fix_words_split_on_two_lines(lines):
  # Input: lines of a text (resulting of a PDF text extraction
  # some lines end with a truncated word and a dash to mark the truncation,
  # the following lines start with remaining of the truncated words
  # Output: the text with the these lines joined

  i = 0
  fixed_lines = list()
  fixed_lines_count = 0
  fixed_line_i = ''

  while i < len(lines) -1:
    if len(fixed_line_i) == 0:
        fixed_line_i = lines[i]

    while re.search('\w\w+-\s*$', fixed_line_i) and re.search('^\s*\w\w+', lines[i+1]):
      line_i =  re.sub('-\s*$', '', fixed_line_i)
      line_i_plus_1 =  re.sub('^\s*', '', lines[i+1])
      fixed_line_i = ''.join([line_i, line_i_plus_1])
      fixed_lines_count += 1
      i+=1
    fixed_lines.append(fixed_line_i)
    fixed_line_i = ''
    i+=1



  if i < len(lines):
    fixed_lines.append(lines[i])
  print ('Debug: len(fixed_lines)',len(fixed_lines),'fixed_lines_count',fixed_lines_count,'len(lines)-fixed_lines_count',len(lines)-fixed_lines_count)
  if len(fixed_lines) != len(lines)-fixed_lines_count:
    print ('Error: len(fixed_lines) != len(lines)-fixed_lines_count', line_i)
    quit()
  return fixed_lines



def segment_pdf_text_extracted (text):
    # input: raw text
    # output: a text division with logical structure (h2-4 and paragraphs), page headers removed and footnotes centralized at the end of the text

    lines = text.split('\n') # .splitlines()

    # remove page headers and centralize footnotes at the end of the text
    trimed_lines, header_left, header_right, footnotes, indent, average_line_len = trim_header_footnotes(lines)

    # add footnotes
    trimed_lines.append('')
    trimed_lines.append('——————')
    trimed_lines.extend(footnotes)

    segmented_lines = segment_into_parag_and_heading_on_layout(trimed_lines)
    segmented_lines = fix_words_split_on_two_lines(segmented_lines)

    # add division level
    segmented_lines.insert(0, '<div>')
    segmented_lines.append('</div>')

    return '\n'.join(segmented_lines)

Debug: is tag <BODY> <re.Match object; span=(0, 6), match='<BODY>'>
Debug: is tag something None
Debug: is BODY tag <re.Match object; span=(0, 5), match='<BODY'>
Debug: generate_LS_id(): -65xolqHLh
Debug: generate_LS_id(): MeAadKwepu
Debug: random_alphanum(12): 4ca2a6231317
Debug: generate_LS_unique_id(): 2f5567be-ce06-e36b-37af-059f1fb0aaea
Debug: generate_LS_unique_id(): 4dc27dde-eb4c-9b62-b629-c1003f21d81e


### 1.2 Load data to process in the input dir

click on the directory symbol on the left to access to it

### 1.3 fix some LS bugs

In [ ]:
import os
from bs4 import BeautifulSoup
#import re


input_dir = 'input'
file_collection_paths = [ f for f in os.listdir(input_dir) if os.path.isfile(os.path.join(input_dir,f)) ]
file_collection_paths.sort()

if len(file_collection_paths) ==0: print ('No file present in the',input_dir,'directory')

!mkdir 'clean'
output_dir = 'clean'

#from nltk import agreement

import json

# Process the whole collection (put a break at the end to test on one document)
for i, file_upload in enumerate(file_collection_paths):
  print ('Processing: file:',i, file_upload)

  with open(input_dir+'/'+file_upload) as f:
    j = json.load(f)
    #print(j)

    # for each task
    for d_index, d in enumerate(j):

      html_content = j[d_index]['data']['text'].strip()
      soup = BeautifulSoup(html_content, 'html.parser')
      if soup.find('div',{'class':'plaintext'}):
        print ('Debug: infering the HTML structure from the layout')
        html_content = '<!DOCTYPE html><html><head></head><body>' + segment_pdf_text_extracted(soup.find('div',{'class':'plaintext'}).get_text()) + '</body></html>'
    #html_content = segment_on_layout(html_content)

      global_text, annotations = parse (html_content)
      global_text = ''.join(global_text)
      #print ('global_text',global_text)

      #p#rint ('j[d_index][annotations]', j[d_index]['annotations'])

      # for each annotator of the task
      for a_index, a in enumerate(j[d_index]['annotations']):
        for r_index, r in enumerate(j[d_index]['annotations'][a_index]['result']):
            #if d_index == 13: print ('Debug:', file_upload, d_index, a_index, r_index, j[d_index]['annotations'][a_index]['result'])
            # Bug: g1.json 13 0 1 [{'id': 'mmZBVhPhUO', similar to g1.json 13 0 0 [{'id': 'mmZBVhPhUO' without globalOffsets
            if 'globalOffsets' in j[d_index]['annotations'][a_index]['result'][r_index]['value']:
              g_start = j[d_index]['annotations'][a_index]['result'][r_index]['value']['globalOffsets']['start']
              g_end = j[d_index]['annotations'][a_index]['result'][r_index]['value']['globalOffsets']['end']
            #old_text = j[d_index]['annotations'][a_index]['result'][r_index]['value']['text']
              new_text = global_text[g_start:g_end]
            #print ('old text:', old_text)
            #print ('new text:', new_text)
            #print ('--')
              j[d_index]['annotations'][a_index]['result'][r_index]['value']['text'] = new_text
    with open(output_dir+'/'+file_upload, 'w') as output_f:
      json.dump(j, output_f) #, indent=4)

mkdir: cannot create directory ‘clean’: File exists
Processing: file: 0 G1.json
Processing: file: 1 G16.json
Processing: file: 2 G2.json
Processing: file: 3 G3.json
Processing: file: 4 G4.json
Processing: file: 5 G5.json
Processing: file: 6 G6.json
Processing: file: 7 G7.json
Processing: file: 8 G8.json
Processing: file: 9 G9.json


### 1.4 turn 'cleaned' LS json to csv

In [ ]:
import os
input_dir = 'clean'
file_collection_paths = [ f for f in os.listdir(input_dir) if os.path.isfile(os.path.join(input_dir,f)) ]
file_collection_paths.sort()
print ('file_collection_paths', file_collection_paths)
if len(file_collection_paths) ==0: print ('No file present in the',input_dir,'directory')

#from nltk import agreement

#my_hash_dict = dict()
#
#def my_hash(k):
#  if len(my_hash_dict) != 0:
#    my_hash_dict[k] = max(my_hash_dict.values()) +1
#  else: my_hash_dict[k] = 1
#  #print ('Debug; my_hash_dict[k]', my_hash_dict[k])
#  return my_hash_dict[k]


category_set = {'Analyse', "Fonction d'annonce", 'Mise en scène', "Sources d'autorité", 'Résolution'}
rhetorical_function_set = {'Présenter le raisonnement de la Cour',
                           'Approuver un argument ou un raisonnement',
                           "Evaluer l'impact de la décision finale",
                           'Citer un extrait',
                           "Fonction d'annonce",
                           'Rejeter un argument/un raisonnement',
                           'Exposer',
                           'Donner des consignes aux juridictions compétentes',
                           'Rappeler',
                           "Accepter de revoir l'affaire",
                           'Donner des consignes aux juridictions inférieures compétentes',  # en plus ?
                           'Décrire le contenu',
                           'Rendre la conclusion de la Cour',
                           'Mentionner' }
type_set = {'', "Décision d'une cour inférieure", "Une source d'autorité non écrite", 'Contexte large', 'Pratiques établies ou normes culturelles', 'Source primaire de droit', 'Décision de la Cour Suprême', 'Un fait jugé ou un élément de procédure', 'Un argument', 'Une décision de la Cour',  'Arguments et prétentions des parties', 'Un problème juridique', 'Décision de la cour suprême', 'Problème juridique', 'Faits jugés', 'Une pratique établie ou norme culturelle', 'Une source secondaire', 'Autres éléments de procédure', 'Source secondaire de droit', 'Une source primaire', 'Une décision passée de la Cour'}
target_set = {'', 'Affaire jugée',    'Autre affaire'}
source_set = {'', 'Du défendeur',  "D'un juge dissident", 'Du requérant'}

import json
#file_name = "g16.json"
max_annotation_values = 0
rows = list()
for file_name in file_collection_paths:
  with open(input_dir+'/'+file_name, encoding='utf-8') as f:
    j = json.load(f)
# Process the whole collection (put a break at the end to test on one document)
# for each task
    for d_index, d in enumerate(j):

      #print ('j[d_index][annotations]', j[d_index]['annotations'])

      all_annotation_keys = set()
      annotator = dict()
      d_id = j[d_index]['id']

      # for each annotator of the task
      for a_index, a in enumerate(j[d_index]['annotations']):

        author_id = j[d_index]['annotations'][a_index]['id']
        first_name = j[d_index]['annotations'][a_index]['completed_by']['first_name']
        email = j[d_index]['annotations'][a_index]['completed_by']['email']
        #print ('first_name', first_name)
        annotations = dict()
        annotation_types = set()
        annotations_list = list()

        text  = j[d_index]['data']['text']


        # for each annotation of the annotator
        for r_index, r in enumerate(j[d_index]['annotations'][a_index]['result']):
          #print (file_name, d_index, a_index, r_index)
          #if d_index == 13: print ('Debug:', file_upload, d_index, a_index, r_index, j[d_index]['annotations'][a_index]['result'])
          # Bug: g1.json 13 0 1 [{'id': 'mmZBVhPhUO', similar to g1.json 13 0 0 [{'id': 'mmZBVhPhUO' without globalOffsets
          if 'globalOffsets' in j[d_index]['annotations'][a_index]['result'][r_index]['value']:
            # get global offsets
            g_start = j[d_index]['annotations'][a_index]['result'][r_index]['value']['globalOffsets']['start']
            g_end = j[d_index]['annotations'][a_index]['result'][r_index]['value']['globalOffsets']['end']
            l_text = j[d_index]['annotations'][a_index]['result'][r_index]['value']['text']
            r_id = j[d_index]['annotations'][a_index]['result'][r_index]['id']

            # get the label (labels or choices) of the current annotion
            if 'labels' in j[d_index]['annotations'][a_index]['result'][r_index]['value']:
              value = j[d_index]['annotations'][a_index]['result'][r_index]['value']['labels'][0]
            else:
              value = j[d_index]['annotations'][a_index]['result'][r_index]['value']['choices'][0]

            # store all the annotations labels and choices at a given position
            # update the list of annotation types encountered
            key = str(g_start)+'_'+str(g_end)
            if key in annotations:
              if value != 'Sentence':
                annotations[key].append(value)
                annotation_types.add(value)
            else:
              if value != 'Sentence':
                annotations[key] = list()
                annotations[key].append(value)
                annotation_types.add(value)
            if value != 'Sentence':
              if len(annotations_list) > 0:
                if annotations_list[len(annotations_list)-1]['start'] == g_start and annotations_list[len(annotations_list)-1]['end'] == g_end:
                  # search for a bug
                  # is there rhetorical function with such values 'Une source primaire'
                  if len(annotations_list[len(annotations_list)-1]['value']) == 1 and value == 'Une source primaire':
                    print ('Debug:',file_name, d_index, a_index, r_index, annotations_list[len(annotations_list)-1]['value'])
                  annotations_list[len(annotations_list)-1]['value'].append(value)
                  max_annotation_values = max(max_annotation_values, len(annotations_list[len(annotations_list)-1]['value']))
                else:
                  current_annotation = dict()
                  current_annotation['start'] = g_start
                  current_annotation['end'] = g_end
                  current_annotation['result_id'] = r_id
                  current_annotation['text'] = l_text #text[g_start:g_end]
                  current_annotation['value'] = list()
                  current_annotation['value'].append(value)
                  max_annotation_values = max(max_annotation_values, len(current_annotation['value']))
                  annotations_list.append(current_annotation)
              else:
                current_annotation = dict()
                current_annotation['start'] = g_start
                current_annotation['end'] = g_end
                current_annotation['result_id'] = r_id
                current_annotation['text'] = l_text # text[g_start:g_end]
                current_annotation['value'] = list()
                current_annotation['value'].append(value)
                max_annotation_values = max(max_annotation_values, len(current_annotation['value']))
                annotations_list.append(current_annotation)
            #old_text = j[d_index]['annotations'][a_index]['result'][r_index]['value']['text']
            #new_text = global_text[g_start:g_end]
            #print ('old text:', old_text)
            #print ('new text:', new_text)
            #print ('--')
            #j[d_index]['annotations'][a_index]['result'][r_index]['value']['text'] = new_text
    #with open(file_upload, 'w') as output_f:
      #json.dump(j, output_f) #, indent=4)


        #print (len(annotations),annotations)
        #print(len(annotation_types),annotation_types)

        #
 #       one_value_annotations = dict()
#        one_value_annotation_types = set()

        #
#        for k in annotations:
#          value = '_'.join(annotations[k])
#          #one_value_annotations[k] = value
#          one_value_annotation_types.add(value)
#          all_annotation_keys.add(k)
        #print (len(one_value_annotations), one_value_annotations)
        #print(len(one_value_annotation_types),one_value_annotation_types)
#        annotator[a_index] = one_value_annotations
      #ordered_annotations_per_annotator = dict()

        for a in annotations_list:
          #print (file_name, d_index, a_index, first_name, email, a['start'], a['end'], a['value'], a['text'])
          curr_a = list()
          curr_a.append(file_name)
          curr_a.append(d_index)
          curr_a.append(d_id)
          curr_a.append(a_index)
          curr_a.append(first_name)
          curr_a.append(email)
          curr_a.append(author_id)
          curr_a.append(a['result_id'])

          curr_a.append(a['start'])
          curr_a.append(a['end'])
          curr_a.append(a['text'])

          new_a_value = [''] * max_annotation_values
          for i in range (0,len(a['value'])):
            if a['value'][i] in category_set:
              new_a_value[0] = a['value'][i]
              if a['value'][i] == "Fonction d'annonce":
                new_a_value[1] = a['value'][i]
            elif a['value'][i] in rhetorical_function_set:
              new_a_value[1] = a['value'][i]
            elif a['value'][i] in type_set:
              new_a_value[2] = a['value'][i]
            elif a['value'][i] in target_set:
                new_a_value[3] = a['value'][i]
            elif a['value'][i] in source_set:
              new_a_value[4] = a['value'][i]
          curr_a.extend(new_a_value)
          #for i in range (0,max_annotation_values):
            #print (i, a['value'][i], len(a['value']), max_annotation_values)
          #  if i < len(a['value']):
          #    curr_a.append(a['value'][i])
          #  else:
          #    curr_a.append('')
          #print (curr_a)
          rows.append (curr_a)

#      ordered_annotations = list()
#      for a_index in annotator:
#          for i, k in enumerate(all_annotation_keys):
#            if not k in annotator[a_index]:
#              #if 'Sentence'
#              ordered_annotations.append([a_index, str(i), str(0)])
#            else:
#              ordered_annotations.append([a_index, str(i), str(my_hash(annotator[a_index][k]))])
#              #print ('debug; annotator[a_index][k]',annotator[a_index][k])
#ordered_annotations



import csv


fields =  ['filename', 'task_index', 'task_id', 'author_index', 'author_firstname', 'author_email', 'author_id',  'result_id', 'start', 'end', 'text',
            'category','rhetorical function','type','target','source']


with open('all_raw_data.csv', 'w', encoding='utf-8') as f:

    # using csv.writer method from CSV package
    write = csv.writer(f)

    write.writerow(fields)
    write.writerows(rows)

file_collection_paths ['G1.json', 'G16.json', 'G2.json', 'G3.json', 'G4.json', 'G5.json', 'G6.json', 'G7.json', 'G8.json', 'G9.json']
Debug: G16.json 4 1 154 ['Analyse']


In [ ]:
print('category', set([row[11] for row in rows]))
print('rhetorical function', set([row[12] for row in rows]))
print('type', set([row[13] for row in rows]))
print('target', set([row[14] for row in rows]))
print('source', set([row[15] for row in rows]))


category {'Mise en scène', "Sources d'autorité", 'Résolution', "Fonction d'annonce", 'Analyse'}
rhetorical function {'Présenter le raisonnement de la Cour', 'Exposer', 'Donner des consignes aux juridictions inférieures compétentes', 'Décrire le contenu', 'Rappeler', 'Rejeter un argument/un raisonnement', 'Mentionner', "Fonction d'annonce", 'Donner des consignes aux juridictions compétentes', "Evaluer l'impact de la décision finale", 'Approuver un argument ou un raisonnement', "Accepter de revoir l'affaire", 'Citer un extrait', 'Rendre la conclusion de la Cour'}
type {'', "Une source d'autorité non écrite", 'Un fait jugé ou un élément de procédure', 'Autres éléments de procédure', 'Une source primaire', 'Problème juridique', 'Un argument', 'Une décision de la Cour', "Décision d'une cour inférieure", 'Une source secondaire', 'Faits jugés', 'Un problème juridique', 'Décision de la Cour Suprême', 'Source primaire de droit', 'Décision de la cour suprême', 'Une décision passée de la Cour', '

## 2- Anas: post-correction of annotation

all_raw_data.csv doesn't content the not annotated sentences

In [ ]:
import pandas as pd

df = pd.read_csv("all_raw_data.csv")
df

,filename,task_index,task_id,author_index,author_firstname,author_email,author_id,result_id,start,end,text,category,rhetorical function,type,target,source
0,G1.json,0,112698080,0,NaN,leo.lisana@etu.univ-nantes.fr,38355122,0Ti7S6BJFk,10634,10688,\n JUSTICE KENNEDY delivered the opinion of t...,Fonction d'annonce,Fonction d'annonce,NaN,NaN,NaN
1,G1.json,0,112698080,0,NaN,leo.lisana@etu.univ-nantes.fr,38355122,dIzMr2Mb-u,10967,11026,The\nUnited States is a contracting state to t...,Mise en scène,Exposer,Contexte large,NaN,NaN
2,G1.json,0,112698080,0,NaN,leo.lisana@etu.univ-nantes.fr,38355122,mK9kf7V0Wu,11027,11176,and\nCongress has implemented its provisions t...,Sources d'autorité,Décrire le contenu,Source primaire de droit,NaN,NaN
3,G1.json,0,112698080,0,NaN,leo.lisana@etu.univ-nantes.fr,38355122,2kmJVRqBYv,11411,11513,The question is\nwhether a parent has a “righ[...,Mise en scène,Exposer,Problème juridique,NaN,NaN
4,G1.json,0,112698080,0,NaN,leo.lisana@etu.univ-nantes.fr,38355122,LUj4TPLrHi,11514,11601,the authority to consent before\nthe other par...,Mise en scène,Exposer,Problème juridique,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27353,G9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,TgL73RqTHE,18689,18884,"Neely held, however, that there are also ...",Analyse,Rappeler,Une décision de la Cour,NaN,NaN
27354,G9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,xRXxkUUY9N,18667,18688,"386 U.S., at .",Sources d'autorité,Mentionner,Décision de la cour suprême,NaN,NaN
27355,G9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,bp4qPr6x3n,13933,13963,"Tr. of Oral Arg. 6, 18, 23.",Sources d'autorité,Mentionner,Source secondaire de droit,NaN,NaN
27356,G9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,-dqqc7Nm1i,13913,13932,"Reply Brief 1, 3-6;",Sources d'autorité,Mentionner,Source secondaire de droit,NaN,NaN


In [ ]:
df.loc[df['rhetorical function'] == 'Donner des consignes aux juridictions inférieures compétentes', 'rhetorical function'] = 'Donner des consignes aux juridictions compétentes'
df.loc[df['type'] == 'Décision de la cour suprême', 'type'] = 'Décision de la Cour Suprême'
df.loc[df['type'] == 'Une décision passée de la Cour', 'type'] = 'Une décision de la Cour'


# df[df["type"]=="Une source d'autorité non écrite"] # Not take the annotation of Mary for this Anomaly later
# the annotation of mary content some anomaly: be attention about how measuring the comparaison between annotations

# Corriger cette annotation: Mise en scène	Exposer	Arguments et prétentions des parties	Affaire jugée (pas de Affaire jugée).
import numpy as np
df.loc[
    (df['category'] == 'Mise en scène') &
    (df['rhetorical function'] == 'Exposer') &
    (df['type'] == 'Arguments et prétentions des parties') &
    (df['target'] == 'Affaire jugée'),
    'target'
] = np.nan

In [ ]:
# Remplacer les valeurs NaN par 'None' pour éviter des erreurs dans les manipulations de chaînes
df.fillna('None', inplace=True)

# Créer une colonne combinant toutes les annotations pour la stratification
df['step'] = (
    df['category'].astype(str) + '_' +
    df['rhetorical function'].astype(str) + '_' +
    df['type'].astype(str) + '_' +
    df['target'].astype(str) + '_' +
    df['source'].astype(str)
)

df.replace('None', np.nan, inplace=True)

In [ ]:
df["step"].value_counts()

,count
step,
Analyse_Présenter le raisonnement de la Cour_None_None_None,3308
Sources d'autorité_Mentionner_Décision de la Cour Suprême_None_None,2883
Mise en scène_Exposer_Faits jugés_None_None,2387
Sources d'autorité_Mentionner_Source primaire de droit_None_None,2292
Analyse_Rappeler_Une décision de la Cour_None_None,2254
Analyse_Rappeler_Une source primaire_None_None,1836
Sources d'autorité_Mentionner_Source secondaire de droit_None_None,1500
Mise en scène_Exposer_Décision d'une cour inférieure_None_None,1242
Analyse_Rappeler_Une pratique établie ou norme culturelle_None_None,1212


In [ ]:
df = df[df['step'] != "Sources d'autorité_Mentionner_None_None_None"]

In [ ]:
df.category.nunique(), df["rhetorical function"].nunique(), df["step"].nunique()

(5, 13, 39)

why we have 39 and not 35? because the annotation of Mary content some errors, and not follow the latest annotation scheme.

In [ ]:
# Sort the sentences within each document
df = df.sort_values(by=['task_id', 'author_index','start']).reset_index(drop=True)

In [ ]:
df.to_csv("all_raw_data.csv", index=False, encoding='utf-8')

## 3- Select the reference for training

In [ ]:
import pandas as pd
#df = pd.read_csv("all_raw_data.csv") #,names=['filename', 'task_index', 'task_id', 'author_index', 'author_firstname', 'author_email', 'author_id',  'result_id', 'start', 'end', 'text', 'category', 'rhetorical function', 'type', 'target', 'source' ])
df

,filename,task_index,task_id,author_index,author_firstname,author_email,author_id,result_id,start,end,text,category,rhetorical function,type,target,source,step
0,G16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,nF76VqHbSX,1332,1386,JUSTICE BREYER delivered the opinion of th...,Fonction d'annonce,Fonction d'annonce,NaN,NaN,NaN,Fonction d'annonce_Fonction d'annonce_None_Non...
1,G16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,as_Y20wPU-,1387,1641,Before seeking a federal writ of habeas corpus...,Sources d'autorité,Décrire le contenu,Source primaire de droit,NaN,NaN,Sources d'autorité_Décrire le contenu_Source p...
2,G16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,PKPbWTc2M3,1642,1752,"Duncan v. Henry, 513 U. S. 364, 365 (1995) (pe...",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
3,G16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,LMCUiBnf3P,1753,2019,"To provide the State with the necessary ""oppor...",Analyse,Rappeler,Une décision de la Cour,NaN,NaN,Analyse_Rappeler_Une décision de la Cour_None_...
4,G16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,3fq1R4NA19,2020,2046,"Duncan, supra, at 365-366;",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27350,G9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,Xda6aaqfBn,18885,18897,"Id., at 326.",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
27351,G9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,oGg5ztWqeY,18898,18942,"We adhere to Neely `s holding and rationale,",Analyse,Présenter le raisonnement de la Cour,NaN,NaN,NaN,Analyse_Présenter le raisonnement de la Cour_N...
27352,G9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,ObtAJTro-h,18943,19189,and today hold that the authority of courts of...,Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...
27353,G9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,qXzDCdhol0,19190,19276,"For the reasons stated, the judgment of the Co...",Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...


In [ ]:
df['filename'] = df['filename'].str.replace('G', 'g')
ref_df = df.loc[((df['filename'] == 'g16.json') & (df['task_index'] <= 13) & (df['author_firstname'] == 'Warren')) |
       ((df['filename'] == 'g16.json') & (df['task_index'] >= 14) & (df['author_firstname'] == 'Mary')) |
        ((df['filename'] == 'g5.json') & (df['task_index'] == 1) & (df['author_email'] == 'leo.lisana@etu.univ-nantes.fr')) |
        ((df['filename'] == 'g5.json') & (df['task_index'] != 1) ) |
        ((df['filename'] == 'g6.json') & (df['task_index'] == 2) & (df['author_email'] == 'leo.lisana@etu.univ-nantes.fr')) |
        ((df['filename'] == 'g6.json') & (df['task_index'] != 2) ) |
        ((df['filename'] == 'g8.json') & (df['task_index'] == 1) & (df['author_email'] == 'leo.lisana@etu.univ-nantes.fr')) |
        ((df['filename'] == 'g8.json') & (df['task_index'] != 1) ) |
        ((df['filename'] == 'g9.json') & (df['task_index'] == 6) & (df['author_email'] == 'leo.lisana@etu.univ-nantes.fr')) |
        ((df['filename'] == 'g9.json') & (df['task_index'] != 6) ) |
       ((df['filename'] != 'g16.json') & (df['filename'] != 'g5.json') & (df['filename'] != 'g6.json') & (df['filename'] != 'g8.json') & (df['filename'] != 'g9.json') ) ]
#df.loc[(df['author_firstname'] == 'Warren')]
ref_df

,filename,task_index,task_id,author_index,author_firstname,author_email,author_id,result_id,start,end,text,category,rhetorical function,type,target,source,step
0,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,nF76VqHbSX,1332,1386,JUSTICE BREYER delivered the opinion of th...,Fonction d'annonce,Fonction d'annonce,NaN,NaN,NaN,Fonction d'annonce_Fonction d'annonce_None_Non...
1,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,as_Y20wPU-,1387,1641,Before seeking a federal writ of habeas corpus...,Sources d'autorité,Décrire le contenu,Source primaire de droit,NaN,NaN,Sources d'autorité_Décrire le contenu_Source p...
2,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,PKPbWTc2M3,1642,1752,"Duncan v. Henry, 513 U. S. 364, 365 (1995) (pe...",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
3,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,LMCUiBnf3P,1753,2019,"To provide the State with the necessary ""oppor...",Analyse,Rappeler,Une décision de la Cour,NaN,NaN,Analyse_Rappeler_Une décision de la Cour_None_...
4,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,3fq1R4NA19,2020,2046,"Duncan, supra, at 365-366;",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27350,g9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,Xda6aaqfBn,18885,18897,"Id., at 326.",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
27351,g9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,oGg5ztWqeY,18898,18942,"We adhere to Neely `s holding and rationale,",Analyse,Présenter le raisonnement de la Cour,NaN,NaN,NaN,Analyse_Présenter le raisonnement de la Cour_N...
27352,g9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,ObtAJTro-h,18943,19189,and today hold that the authority of courts of...,Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...
27353,g9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,qXzDCdhol0,19190,19276,"For the reasons stated, the judgment of the Co...",Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...


In [ ]:
ref_df[ref_df.duplicated(subset=['task_index', 'author_email'], keep=False)]

,filename,task_index,task_id,author_index,author_firstname,author_email,author_id,result_id,start,end,text,category,rhetorical function,type,target,source,step
0,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,nF76VqHbSX,1332,1386,JUSTICE BREYER delivered the opinion of th...,Fonction d'annonce,Fonction d'annonce,NaN,NaN,NaN,Fonction d'annonce_Fonction d'annonce_None_Non...
1,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,as_Y20wPU-,1387,1641,Before seeking a federal writ of habeas corpus...,Sources d'autorité,Décrire le contenu,Source primaire de droit,NaN,NaN,Sources d'autorité_Décrire le contenu_Source p...
2,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,PKPbWTc2M3,1642,1752,"Duncan v. Henry, 513 U. S. 364, 365 (1995) (pe...",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
3,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,LMCUiBnf3P,1753,2019,"To provide the State with the necessary ""oppor...",Analyse,Rappeler,Une décision de la Cour,NaN,NaN,Analyse_Rappeler_Une décision de la Cour_None_...
4,g16.json,0,110895127,0,Warren,warren.bonnard@univ-nantes.fr,37628600,3fq1R4NA19,2020,2046,"Duncan, supra, at 365-366;",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27350,g9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,Xda6aaqfBn,18885,18897,"Id., at 326.",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
27351,g9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,oGg5ztWqeY,18898,18942,"We adhere to Neely `s holding and rationale,",Analyse,Présenter le raisonnement de la Cour,NaN,NaN,NaN,Analyse_Présenter le raisonnement de la Cour_N...
27352,g9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,ObtAJTro-h,18943,19189,and today hold that the authority of courts of...,Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...
27353,g9.json,17,122986651,0,NaN,leo.lisana@etu.univ-nantes.fr,41384572,qXzDCdhol0,19190,19276,"For the reasons stated, the judgment of the Co...",Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...


In [ ]:
# Grouper par 'task_id' et obtenir les auteurs uniques pour chaque tâche
task_authors = ref_df.groupby('task_id')['author_email'].unique()

# Filtrer pour obtenir les task_id avec plus d'un auteur
multiple_authors_tasks = task_authors[task_authors.apply(len) > 1]

# Obtenir les informations pour chaque task_id avec plusieurs auteurs
tasks_info = ref_df[ref_df['task_id'].isin(multiple_authors_tasks.index)]

# Grouper et obtenir les informations spécifiques
task_details = tasks_info.groupby('task_id').agg({
    'filename': 'first',  # Prendre le premier 'filename' rencontré pour chaque groupe
    'task_index': 'first',  # Prendre le premier 'task_index' rencontré pour chaque groupe
    'author_email': lambda x: set(x)  # Ensemble d'emails d'auteurs pour éviter les duplications
}).reset_index()

In [ ]:
task_details

,task_id,filename,task_index,author_email


In [ ]:
ref_df.category.nunique(), ref_df["rhetorical function"].nunique(), ref_df["step"].nunique()

(5, 13, 35)

yeeeees! 35 steps !!!!

In [ ]:
ref_df.to_csv("ref_data.csv", index=False, encoding="utf-8")

## 4- Adding Meta-data

In [ ]:
import pandas as pd
ref_df = pd.read_csv("/content/drive/MyDrive/Thèse/Thèse - Anas Belfathi/WP2/annotated-data-preprocessing/csv/ref_data.csv")

In [ ]:
metadata_df = pd.read_csv("/content/drive/MyDrive/Thèse/Thèse - Anas Belfathi/WP2/corpus-construction/data/all-opinions-metadata.csv")
metadata_df

,author_name,category,per_curiam,case_name,date_filed,federal_cite_one,absolute_url,cluster,year_filed,scdb_id,...,text,word_count,topics,topics_name,group_id,opinions_html,opinions_text,cited_by,summary,related_opinions
0,Justice Black,majority,False,Howitt v. United States,1946-05-06,328 U.S. 189,https://www.courtlistener.com/opinion/104292/h...,https://www.courtlistener.com/api/rest/v3/clus...,1946,1945-095,...,The wartime transportation shortage during the...,584,16,"carrier, rate",1,"\n<!DOCTYPE html>\n\n<html lang=""en"">\n<head>\...","<div id=""opinion-content"">\n<div class=""serif-...","<ul>\n<li>\n<a href=""/opinion/106619/united-st...",NaN,NaN
1,Justice Breyer,majority,False,Mayo Collaborative Services v. Prometheus Labo...,2012-03-20,NaN,https://www.courtlistener.com/opinion/625710/m...,https://www.courtlistener.com/api/rest/v3/clus...,2012,2011-034,...,Section 101 of the Patent Act defines patentab...,7391,11,"patent, invention",1,"\n<!DOCTYPE html>\n\n<html lang=""en"">\n<head>\...","<div id=""opinion-content"">\n<div class=""plaint...","<ul>\n<li>\n<a href=""/opinion/626929/la-printe...",<ul>\n<li>\n “[S]imply appendin...,"<ul>\n<li>\n<a href=""/opinion/181305/prometheu..."
2,Justice Burger,majority,False,NLRB v. Catholic Bishop of Chicago,1979-03-21,NaN,https://www.courtlistener.com/opinion/110040/n...,https://www.courtlistener.com/api/rest/v3/clus...,1979,1978-060,...,This case arises out of the National Labor Rel...,4652,7,"school, child",1,"\n<!DOCTYPE html>\n\n<html lang=""en"">\n<head>\...","<div id=""opinion-content"">\n<div class=""serif-...","<ul>\n<li>\n<a href=""/opinion/2620886/steel-co...","<ul>\n<li>\n Holding that ""[i]t...","<ul>\n<li>\n<a href=""/opinion/1398134/vitello-..."
3,Justice Burton,majority,False,Boutell v. Walling,1946-02-25,327 U.S. 463,https://www.courtlistener.com/opinion/104255/b...,https://www.courtlistener.com/api/rest/v3/clus...,1946,1945-093,...,This suit was brought in the District Court of...,1577,16,"carrier, rate",1,"\n<!DOCTYPE html>\n\n<html lang=""en"">\n<head>\...","<div id=""opinion-content"">\n<div class=""serif-...","<ul>\n<li>\n<a href=""/opinion/107169/idaho-she...",<ul>\n<li>\n Involving prior co...,"<ul>\n<li>\n<a href=""/opinion/8013759/germania..."
4,Justice Clark,majority,False,Corn Products Refining Co. v. Commissioner,1956-01-09,NaN,https://www.courtlistener.com/opinion/105327/c...,https://www.courtlistener.com/api/rest/v3/clus...,1956,1955-005,...,This case concerns the tax treatment to be acc...,1740,14,"price, commerce",1,"\n<!DOCTYPE html>\n\n<html lang=""en"">\n<head>\...","<div id=""opinion-content"">\n<div class=""serif-...","<ul>\n<li>\n<a href=""/opinion/105668/commissio...",NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
355,Justice Scalia,majority,False,Reno v. American-Arab Anti-Discrimination Comm.,1999-02-24,NaN,https://www.courtlistener.com/opinion/118264/r...,https://www.courtlistener.com/api/rest/v3/clus...,1999,1998-024,...,[]\nRespondents sued petitioners for allegedl...,4877,15,"alien, deportation",20,"\n<!DOCTYPE html>\n\n<html lang=""en"">\n<head>\...","<div id=""opinion-content"">\n<div class=""serif-...","<ul>\n<li>\n<a href=""/opinion/118452/ins-v-st-...","<ul>\n<li>\n Stating that ""prot...","<ul>\n<li>\n<a href=""/opinion/7821212/shehadeh..."
356,Justice Sotomayor,majority,False,Brumfield v. Cain,2015-06-18,NaN,https://www.courtlistener.com/opinion/2809767/...,https://www.courtlistener.com/api/rest/v3/clus...,2015,NaN,...,"In Atkins v. Virginia, 536 U.S. 304 (2002), th...",5947,4,"agency, government",20,"\n<!DOCTYPE html>\n\n<html lang=""en"">\n<head>\...","<div id=""opinion-content"">\n<div class=""plaint...","<ul class=""dropdown-menu"">\n<li><a href=""/help...",NaN,"<ul>\n<li>\n<a href=""/opinion/2811850/brumfiel..."
357,Justice Stevens,majority,False,"Fort Gratiot Sanitary Landfill, Inc. v. Michig...",1992-06-01,NaN,https://www.courtlistener.com/opinion/112741/f...,https://www.c

In [ ]:
import pandas as pd
import re

# Étape 1 : Extraire les premières phrases pour chaque document (task_id) dans un groupe (filename)
def extract_first_sentences(df_annotations):
    first_sentences = df_annotations.groupby(['filename', 'task_id']).apply(
        lambda group: group.loc[group['start'].idxmin()]['text']
    ).reset_index(name='first_sentence')
    # Convertir les phrases en minuscules
    first_sentences['first_sentence'] = first_sentences['first_sentence'].str.lower()
    # Extraire le numéro de group_id depuis le filename
    first_sentences['group_id'] = first_sentences['filename'].str.extract(r'(\d+)').astype(int)
    return first_sentences

# Étape 2 : Correspondance stricte entre les premières phrases et les noms d'auteurs (priorité au premier mentionné)
def match_and_add_metadata_to_annotations(df_annotations, df_metadata, required_metadata_columns):
    # Extraire les premières phrases
    first_sentences = extract_first_sentences(df_annotations)

    # Liste pour stocker les résultats alignés
    merged_results = []

    # Itérer sur chaque groupe (group_id extrait)
    for group_id in first_sentences['group_id'].unique():
        # Filtrer les métadonnées pour ce group_id
        group_metadata = df_metadata[df_metadata['group_id'] == group_id]

        # Obtenir les noms d'auteurs et les convertir en minuscules
        author_names = group_metadata['author_name'].str.lower().tolist()

        # Obtenir les premières phrases du groupe
        sentences_in_group = first_sentences[first_sentences['group_id'] == group_id]

        # Vérifier les correspondances des noms d'auteurs avec les premières phrases
        for _, row in sentences_in_group.iterrows():
            sentence = row['first_sentence']

            # Trouver les auteurs mentionnés dans l'ordre d'apparition
            matched_authors = []
            for author in author_names:
                if re.search(rf'\b{re.escape(author)}\b', sentence):
                    matched_authors.append(author)

            # Considérer uniquement le premier auteur mentionné
            if matched_authors:
                first_author = matched_authors[0]
                metadata_subset = group_metadata[group_metadata['author_name'].str.lower() == first_author]
                for _, meta_row in metadata_subset.iterrows():
                    merged_results.append({
                        'filename': row['filename'],
                        'task_id': row['task_id'],
                        'text': row['first_sentence'],
                        **{col: meta_row[col] for col in required_metadata_columns}  # Ajouter uniquement les colonnes nécessaires
                    })

    # Retourner le DataFrame des annotations enrichi
    return pd.DataFrame(merged_results).drop_duplicates()

# Colonnes nécessaires depuis les métadonnées
required_metadata_columns = ['author_name', 'case_name', "absolute_url", 'year_filed', 'topics_name']

# Appel de la fonction pour ajouter les métadonnées nécessaires
result_df = match_and_add_metadata_to_annotations(ref_df, metadata_df, required_metadata_columns)

# Afficher les résultats
# result_df.head()


<ipython-input-40-c28d414f7f5b>:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_sentences = df_annotations.groupby(['filename', 'task_id']).apply(


In [ ]:
result_df

,filename,task_id,text,author_name,case_name,absolute_url,year_filed,topics_name
0,g1.json,112698080,\n justice kennedy delivered the opinion of t...,Justice Kennedy,Abbott v. Abbott,https://www.courtlistener.com/opinion/146554/a...,2010,"school, child"
1,g1.json,112698081,"justice o'connor, delivered the opinion of the...",Justice O'Connor,Beecham v. United States,https://www.courtlistener.com/opinion/1087963/...,1994,"offense, conviction"
2,g1.json,112698082,mr. justice burton delivered the opinion of th...,Justice Burton,Boutell v. Walling,https://www.courtlistener.com/opinion/104255/b...,1946,"carrier, rate"
3,g1.json,112698083,mr. justice clark delivered the opinion of...,Justice Clark,Corn Products Refining Co. v. Commissioner,https://www.courtlistener.com/opinion/105327/c...,1956,"price, commerce"
4,g1.json,112698084,"justice ginsburg, delivered the opinion of the...",Justice Ginsburg,"Doctor's Associates, Inc. v. Casarotto",https://www.courtlistener.com/opinion/118029/d...,1996,"arbitration, agreement"
...,...,...,...,...,...,...,...,...
175,g9.json,122986647,\n justice sotomayor delivered the opinion of...,Justice Sotomayor,Samsung Electronics Co. v. Apple Inc.,https://www.courtlistener.com/opinion/4327561/...,2016,"patent, invention"
176,g9.json,122986648,"justice white, delivered the opinion of the co...",Justice White,United States v. Alaska,https://www.courtlistener.com/opinion/112722/u...,1992,"land, water"
177,g9.json,122986649,\n justice breyer delivered the opinion of t...,Justice Breyer,United States v. Marcus,https://www.courtlistener.com/opinion/146982/u...,2010,"offense, conviction"
178,g9.json,122986650,chief justice rehnquist delivered the opinion ...,Justice Rehnquist,United States v. Ramirez,https://www.courtlistener.com/opinion/118180/u...,1998,"search, officer"


In [ ]:
import pandas as pd

# Ajout des colonnes de métadonnées à côté de `task_id` dans `ref_df`
def merge_metadata_with_annotations(ref_df, result_df):
    # Sélection des colonnes nécessaires dans result_df pour éviter de dupliquer des données inutiles
    metadata_columns = ['filename', 'task_id', 'author_name', 'case_name', 'absolute_url', 'year_filed', 'topics_name']
    result_df = result_df[metadata_columns]

    # Merge des DataFrames en fonction des colonnes communes
    merged_df = ref_df.merge(result_df, on=['filename', 'task_id'], how='left')

    # Identifier les colonnes non nécessaires (celles ajoutées en double lors du merge)
    unnecessary_cols = [col for col in metadata_columns if col in ref_df.columns and col not in ['filename', 'task_id']]

    # Supprimer les colonnes en double, si elles existent
    merged_df = merged_df.drop(columns=unnecessary_cols, errors='ignore')

    # Réorganiser les colonnes pour placer les métadonnées juste après `task_id`
    cols = list(merged_df.columns)
    task_id_index = cols.index('task_id')
    reordered_cols = (
        cols[:task_id_index + 1]  # Colonnes avant et incluant `task_id`
        + [col for col in metadata_columns if col not in ['filename', 'task_id']]  # Colonnes des métadonnées
        + [col for col in cols[task_id_index + 1:] if col not in metadata_columns]  # Colonnes restantes
    )
    merged_df = merged_df[reordered_cols]

    return merged_df

# Appel de la fonction pour ajouter les métadonnées
final_df = merge_metadata_with_annotations(ref_df, result_df)

# Afficher le DataFrame final avec les métadonnées ajoutées après `task_id`
# final_df.head()


In [ ]:
# import pandas as pd

# # Ajout des colonnes de métadonnées à côté de `task_id` dans `ref_df`
# def merge_metadata_with_annotations(ref_df, result_df):
#     # Sélection des colonnes nécessaires dans result_df pour éviter de dupliquer des données inutiles
#     metadata_columns = ['filename', 'task_id', 'author_name', 'case_name', 'absolute_url', 'year_filed', 'topics_name']
#     result_df = result_df[metadata_columns]

#     # Merge des DataFrames en fonction des colonnes communes
#     merged_df = ref_df.merge(result_df, on=['filename', 'task_id'], how='left')

#     # Réorganiser les colonnes pour placer les métadonnées juste après `task_id`
#     cols = list(merged_df.columns)
#     task_id_index = cols.index('task_id')
#     reordered_cols = (
#         cols[:task_id_index + 1]  # Colonnes avant et incluant `task_id`
#         + [col for col in metadata_columns if col not in ['filename', 'task_id']]  # Colonnes des métadonnées
#         + cols[task_id_index + 1:]  # Colonnes restantes après `task_id`
#     )
#     merged_df = merged_df[reordered_cols]

#     return merged_df

# # Appel de la fonction pour ajouter les métadonnées
# final_df = merge_metadata_with_annotations(ref_df, result_df)

# # Afficher le DataFrame final avec les métadonnées ajoutées après `task_id`
# # final_df.head()


In [ ]:
final_df

,filename,task_index,task_id,author_name,case_name,absolute_url,year_filed,topics_name,author_index,author_firstname,...,result_id,start,end,text,category,rhetorical function,type,target,source,step
0,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,nF76VqHbSX,1332,1386,JUSTICE BREYER delivered the opinion of th...,Fonction d'annonce,Fonction d'annonce,NaN,NaN,NaN,Fonction d'annonce_Fonction d'annonce_None_Non...
1,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,as_Y20wPU-,1387,1641,Before seeking a federal writ of habeas corpus...,Sources d'autorité,Décrire le contenu,Source primaire de droit,NaN,NaN,Sources d'autorité_Décrire le contenu_Source p...
2,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,PKPbWTc2M3,1642,1752,"Duncan v. Henry, 513 U. S. 364, 365 (1995) (pe...",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
3,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,LMCUiBnf3P,1753,2019,"To provide the State with the necessary ""oppor...",Analyse,Rappeler,Une décision de la Cour,NaN,NaN,Analyse_Rappeler_Une décision de la Cour_None_...
4,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,3fq1R4NA19,2020,2046,"Duncan, supra, at 365-366;",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26322,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,Xda6aaqfBn,18885,18897,"Id., at 326.",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
26323,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,oGg5ztWqeY,18898,18942,"We adhere to Neely `s holding and rationale,",Analyse,Présenter le raisonnement de la Cour,NaN,NaN,NaN,Analyse_Présenter le raisonnement de la Cour_N...
26324,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,ObtAJTro-h,18943,19189,and today hold that the authority of courts of...,Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...
26325,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,qXzDCdhol0,19190,19276,"For the reasons stated, the judgment of the Co...",Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...


In [ ]:
final_df.to_csv("ref_with_metadata.csv", index=False, encoding="utf-8")

## 5- Turn labels from FR to EN

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Thèse/Thèse - Anas Belfathi/WP2/annotated-data-preprocessing/csv/ref_with_metadata.csv")
df

,filename,task_index,task_id,author_name,case_name,absolute_url,year_filed,topics_name,author_index,author_firstname,...,result_id,start,end,text,category,rhetorical function,type,target,source,step
0,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,nF76VqHbSX,1332,1386,JUSTICE BREYER delivered the opinion of th...,Fonction d'annonce,Fonction d'annonce,NaN,NaN,NaN,Fonction d'annonce_Fonction d'annonce_None_Non...
1,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,as_Y20wPU-,1387,1641,Before seeking a federal writ of habeas corpus...,Sources d'autorité,Décrire le contenu,Source primaire de droit,NaN,NaN,Sources d'autorité_Décrire le contenu_Source p...
2,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,PKPbWTc2M3,1642,1752,"Duncan v. Henry, 513 U. S. 364, 365 (1995) (pe...",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
3,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,LMCUiBnf3P,1753,2019,"To provide the State with the necessary ""oppor...",Analyse,Rappeler,Une décision de la Cour,NaN,NaN,Analyse_Rappeler_Une décision de la Cour_None_...
4,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,3fq1R4NA19,2020,2046,"Duncan, supra, at 365-366;",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26322,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,Xda6aaqfBn,18885,18897,"Id., at 326.",Sources d'autorité,Mentionner,Décision de la Cour Suprême,NaN,NaN,Sources d'autorité_Mentionner_Décision de la C...
26323,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,oGg5ztWqeY,18898,18942,"We adhere to Neely `s holding and rationale,",Analyse,Présenter le raisonnement de la Cour,NaN,NaN,NaN,Analyse_Présenter le raisonnement de la Cour_N...
26324,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,ObtAJTro-h,18943,19189,and today hold that the authority of courts of...,Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...
26325,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,qXzDCdhol0,19190,19276,"For the reasons stated, the judgment of the Co...",Résolution,Rendre la conclusion de la Cour,NaN,NaN,NaN,Résolution_Rendre la conclusion de la Cour_Non...


In [ ]:
## save with utf-8 encoding

df.to_csv("ref_with_metadata.csv", encoding="utf-8")

In [ ]:
unique_values = {
    'category': df['category'].unique().tolist(),
    'rhetorical_function': df['rhetorical function'].unique().tolist(),
    'type': df['type'].dropna().unique().tolist(),  # Dropping NaN values
    'target': df['target'].dropna().unique().tolist(),  # Dropping NaN values
    'source': df['source'].dropna().unique().tolist()  # Dropping NaN values
}

unique_values

{'category': ["Fonction d'annonce",
  "Sources d'autorité",
  'Analyse',
  'Mise en scène',
  'Résolution'],
 'rhetorical_function': ["Fonction d'annonce",
  'Décrire le contenu',
  'Mentionner',
  'Rappeler',
  'Exposer',
  "Accepter de revoir l'affaire",
  'Présenter le raisonnement de la Cour',
  'Rejeter un argument/un raisonnement',
  'Rendre la conclusion de la Cour',
  'Donner des consignes aux juridictions compétentes',
  'Approuver un argument ou un raisonnement',
  'Citer un extrait',
  "Evaluer l'impact de la décision finale"],
 'type': ['Source primaire de droit',
  'Décision de la Cour Suprême',
  'Une décision de la Cour',
  'Problème juridique',
  'Faits jugés',
  'Source secondaire de droit',
  "Décision d'une cour inférieure",
  'Autres éléments de procédure',
  'Un fait jugé ou un élément de procédure',
  'Une pratique établie ou norme culturelle',
  'Une source primaire',
  'Un argument',
  'Arguments et prétentions des parties',
  'Contexte large',
  'Un problème ju

In [ ]:
import pandas as pd

# Créer les dictionnaires de mapping (en fonction de votre tableau de correspondances)
mapping_category = {
    "Fonction d'annonce": "Announcing",
    "Mise en scène": "Setting the scene",
    "Sources d'autorité": "Sources of authority",
    "Analyse": "Analysis",
    "Résolution": "Resolution"
}

mapping_rhetorical_function = {
    "Fonction d'annonce": "Announcing",
    "Exposer": "Presenting jurisdiction",
    "Décrire le contenu": "Describing",
    "Mentionner": "Quoting",
    "Rappeler": "Recalling",
    "Citer un extrait": "Citing",
    "Présenter le raisonnement de la Cour": "Stating the Court’s reasoning",
    "Rejeter un argument/un raisonnement": "Rejecting arguments/a reasoning",
    "Approuver un argument ou un raisonnement": "Accepting arguments/a reasoning",
    "Accepter de revoir l'affaire": "Granting certiorari",
    "Donner des consignes aux juridictions compétentes": "Giving instructions to competent courts",
    "Rendre la conclusion de la Cour": "Giving the holding of the Court",
    "Evaluer l'impact de la décision finale": "Evaluating the impact of the decision",

}

mapping_type = {
  'Contexte large' : "Context",
  'Source primaire de droit': "Primary source of law",
  'Problème juridique': "Legal question(s)",
  'Faits jugés' : "Adjudicated facts",
  'Source secondaire de droit': "Secondary source of law",
  'Pratiques établies ou normes culturelles': "Established practices or cultural norms",
  'Autres éléments de procédure': "Other procedural events",
  "Décision d'une cour inférieure": "Lower court decision",
  'Décision de la Cour Suprême': "SCOTUS decision",
  'Un fait jugé ou un élément de procédure': "An adjudicated fact or procedural event",
  'Un problème juridique': "Legal question(s)",
  'Une source primaire': "A primary source",
  'Une source secondaire': "A secondary source",
  'Une pratique établie ou norme culturelle': "An established practice or cultural norm",
  'Un argument': "An argument",
  'Une décision de la Cour' : "SCOTUS decision",
  'Arguments et prétentions des parties': "Parties' legal claims and arguments"
}



mapping_target = {
    'Affaire jugée': "Present case",
    'Autre affaire': "Another case"
}

mapping_source = {
    "Du défendeur": "Respondent",
    "Du requérant": "Petitioner",
    "D'un juge dissident": "Dissenting justice(s)"
}

# Appliquer les mappings sur votre DataFrame
# df['category'] = df['category'].map(mapping_category)
# df['rhetorical function'] = df['rhetorical function'].map(mapping_rhetorical_function)
# df['type'] = df['type'].map(mapping_type)
# df['target'] = df['target'].map(mapping_target)
# df['source'] = df['source'].map(mapping_source)

In [ ]:
# Fonction générale pour traduire les colonnes
def translate_labels(df, direction='fr_to_en'):
    # Inverser les mappings si nécessaire
    if direction == 'en_to_fr':
        mappings = {
            'category': {v: k for k, v in mapping_category.items()},
            'rhetorical function': {v: k for k, v in mapping_rhetorical_function.items()},
            'type': {v: k for k, v in mapping_type.items()},
            'target': {v: k for k, v in mapping_target.items()},
            'source': {v: k for k, v in mapping_source.items()},
        }
    elif direction == 'fr_to_en':
        mappings = {
            'category': mapping_category,
            'rhetorical function': mapping_rhetorical_function,
            'type': mapping_type,
            'target': mapping_target,
            'source': mapping_source,
        }
    else:
        raise ValueError("Direction must be 'fr_to_en' or 'en_to_fr'")

    # Traduire chaque colonne
    for column, mapping in mappings.items():
        if column in df.columns:
            df[column] = df[column].map(mapping).fillna(df[column])  # Préserver les valeurs non trouvées

    def create_step(row):
      # Créer la colonne 'step' en joignant les valeurs avec un "_", en ignorant les NaN
      elements = [row['category'], row['rhetorical function'], row['type'], row['target'], row['source']]
      step = '_'.join([str(e) for e in elements if pd.notna(e)])  # Ne garder que les non-NaN
      return step


    df['step'] = df.apply(create_step, axis=1)

    return df

# Exemple d'utilisation
# Traduire de français à anglais
df = translate_labels(df, direction='fr_to_en')

# Traduire d'anglais à français
# df = translate_labels(df, direction='en_to_fr')

In [ ]:
unique_values_en = {
    'category': df['category'].unique().tolist(),
    'rhetorical_function': df['rhetorical function'].unique().tolist(),
    'type': df['type'].dropna().unique().tolist(),  # Dropping NaN values
    'target': df['target'].dropna().unique().tolist(),  # Dropping NaN values
    'source': df['source'].dropna().unique().tolist()  # Dropping NaN values
}

unique_values_en

{'category': ['Announcing',
  'Sources of authority',
  'Analysis',
  'Setting the scene',
  'Resolution'],
 'rhetorical_function': ['Announcing',
  'Describing',
  'Quoting',
  'Recalling',
  'Presenting jurisdiction',
  'Granting certiorari',
  'Stating the Court’s reasoning',
  'Rejecting arguments/a reasoning',
  'Giving the holding of the Court',
  'Giving instructions to competent courts',
  'Accepting arguments/a reasoning',
  'Citing',
  'Evaluating the impact of the decision'],
 'type': ['Primary source of law',
  'SCOTUS decision',
  'Legal question(s)',
  'Adjudicated facts',
  'Secondary source of law',
  'Lower court decision',
  'Other procedural events',
  'An adjudicated fact or procedural event',
  'An established practice or cultural norm',
  'A primary source',
  'An argument',
  "Parties' legal claims and arguments",
  'Context',
  'A secondary source',
  'Established practices or cultural norms'],
 'target': ['Present case', 'Another case'],
 'source': ['Respondent

In [ ]:
df

,filename,task_index,task_id,author_name,case_name,absolute_url,year_filed,topics_name,author_index,author_firstname,...,result_id,start,end,text,category,rhetorical function,type,target,source,step
0,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,nF76VqHbSX,1332,1386,JUSTICE BREYER delivered the opinion of th...,Announcing,Announcing,NaN,NaN,NaN,Announcing_Announcing
1,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,as_Y20wPU-,1387,1641,Before seeking a federal writ of habeas corpus...,Sources of authority,Describing,Primary source of law,NaN,NaN,Sources of authority_Describing_Primary source...
2,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,PKPbWTc2M3,1642,1752,"Duncan v. Henry, 513 U. S. 364, 365 (1995) (pe...",Sources of authority,Quoting,SCOTUS decision,NaN,NaN,Sources of authority_Quoting_SCOTUS decision
3,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,LMCUiBnf3P,1753,2019,"To provide the State with the necessary ""oppor...",Analysis,Recalling,SCOTUS decision,NaN,NaN,Analysis_Recalling_SCOTUS decision
4,g16.json,0,110895127,Justice Breyer,Baldwin v. Reese,https://www.courtlistener.com/opinion/134723/b...,2004,"offense, conviction",0,Warren,...,3fq1R4NA19,2020,2046,"Duncan, supra, at 365-366;",Sources of authority,Quoting,SCOTUS decision,NaN,NaN,Sources of authority_Quoting_SCOTUS decision
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26322,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,Xda6aaqfBn,18885,18897,"Id., at 326.",Sources of authority,Quoting,SCOTUS decision,NaN,NaN,Sources of authority_Quoting_SCOTUS decision
26323,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,oGg5ztWqeY,18898,18942,"We adhere to Neely `s holding and rationale,",Analysis,Stating the Court’s reasoning,NaN,NaN,NaN,Analysis_Stating the Court’s reasoning
26324,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,ObtAJTro-h,18943,19189,and today hold that the authority of courts of...,Resolution,Giving the holding of the Court,NaN,NaN,NaN,Resolution_Giving the holding of the Court
26325,g9.json,17,122986651,Justice Ginsburg,Weisgram v. Marley Co.,https://www.courtlistener.com/opinion/118337/w...,2000,"plaintiff, immunity",0,NaN,...,qXzDCdhol0,19190,19276,"For the reasons stated, the judgment of the Co...",Resolution,Giving the holding of the Court,NaN,NaN,NaN,Resolution_Giving the holding of the Court


In [ ]:
df.to_csv("ref_with_metadata_en.csv", index=False)